
# Rwanda Tourism Assistant using Qwen and Agentic Hybrid RAG

## 1. Project Overview

### Objective
To develop an intelligent tourism assistant that provides accurate,
relevant and informative answers about tourism destinations in Rwanda.

The system will use Retrieval-Augmented Generation (RAG) to retrieve
information from a Rwanda tourism knowledge base before generating
answers using a Large Language Model.

### Main Technologies

- Python: Main programming language
- Qwen: Large Language Model
- BGE-M3: Text embedding model
- ChromaDB: Vector database
- Neo4j: Graph database
- LangGraph: Agent orchestration
- Google Colab: Development environment

### Main Architecture

1. Data collection and preparation
2. Data cleaning and preprocessing
3. Vector database development
4. Knowledge graph development
5. Qwen integration
6. Agentic RAG development
7. Hybrid retrieval
8. System evaluation
9. Final demonstration


## 2. Environment Setup

In this section, we inspect the Google Colab environment.

We check the Python version and available hardware resources
before installing the libraries required for our project.

In [ ]:

# Import Python's system library
import sys

# Import PyTorch to check GPU availability
import torch

# Check Python version
print("Python Version:", sys.version)

# Check PyTorch version
print("PyTorch Version:", torch.__version__)

# Check whether a GPU is available
print("GPU Available:", torch.cuda.is_available())

# Display GPU information if available
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. We are currently using CPU.")


# 02. Dataset Loading

In this section, we will load our Rwanda tourism dataset into Python.

Our dataset is stored in JSON format. JSON allows us to organize information using objects, arrays, and key-value pairs.

We will:
- Upload the JSON file.
- Load it into Python.
- Check whether it was loaded successfully.
- Inspect the main sections of the dataset.

In [ ]:
# Import Python's JSON library
import json

# Import Colab's file upload functionality
from google.colab import files

# Upload our JSON dataset
uploaded = files.upload()

# Get the uploaded filename
file_name = list(uploaded.keys())[0]

# Open and read the JSON file
with open(file_name, "r", encoding="utf-8") as file:
    tourism_data = json.load(file)

# Confirm successful loading
print("Dataset loaded successfully!")
print("File name:", file_name)

# Display the main sections
print("\nMain sections:", tourism_data.keys())

### 2.1 Checking the number of destinations

We will access the `places` section and count how many tourism destinations are stored in our dataset.

This helps us confirm that our dataset contains the expected records.

In [ ]:
# Access the places section
places = tourism_data["places"]

# Count the number of destinations
total_places = len(places)

print("Total tourism destinations:", total_places)

In [ ]:
# Display the first tourism destination
first_place = places[0]

print(json.dumps(first_place, indent=4, ensure_ascii=False))

In [ ]:
# Display all fields available in a destination record
print("Available fields:\n")

for field in first_place.keys():
    print("-", field)

### 2.2 Data Quality Assessment

In this section, we will inspect our dataset for missing information and duplicate records.

This is important because incomplete or duplicated data can affect retrieval quality and produce unreliable answers.

In [ ]:
from collections import Counter

# 1. Check for missing or empty values
missing_values = {}

for field in places[0].keys():
    count = 0

    for place in places:
        value = place.get(field)

        if value is None or value == "" or value == [] or value == {}:
            count += 1

    missing_values[field] = count

print("MISSING VALUES PER FIELD")
print("-" * 35)

for field, count in missing_values.items():
    print(f"{field}: {count}")


# 2. Check for duplicate destination IDs
ids = [place["id"] for place in places]
duplicate_ids = [item for item, count in Counter(ids).items() if count > 1]

print("\nDUPLICATE IDs:", duplicate_ids if duplicate_ids else "None")


# 3. Check for duplicate destination names
names = [place["name"] for place in places]
duplicate_names = [
    item for item, count in Counter(names).items() if count > 1
]

print("DUPLICATE NAMES:", duplicate_names if duplicate_names else "None")


# 4. Count destinations by type
types = Counter(place["type"] for place in places)

print("\nDESTINATIONS BY TYPE")
for place_type, count in types.items():
    print(f"{place_type}: {count}")


# 5. Count destinations by region
regions = Counter(place["region"] for place in places)

print("\nDESTINATIONS BY REGION")
for region, count in regions.items():
    print(f"{region}: {count}")

### 2.3 Identifying incomplete destination information

We found that 13 destinations have empty tips.

Let's identify these destinations so we can keep track of information that may need to be collected from reliable sources later.

In [ ]:
# Find destinations with empty tips
places_without_tips = [
    place["name"]
    for place in places
    if not place.get("tips")
]

print("Destinations without tips:")
print("-" * 40)

for name in places_without_tips:
    print(name)

print("\nTotal:", len(places_without_tips))

### 3.1 Creating a cleaned dataset

We will create a separate copy of our original tourism dataset.

We will perform basic text cleaning while preserving the original information.

The raw dataset will remain unchanged so that we can always compare it with our cleaned version.

In [ ]:
import copy

# Create a separate copy of our original dataset
cleaned_data = copy.deepcopy(tourism_data)

# Access the copied destination records
cleaned_places = cleaned_data["places"]


# Clean text values without changing their meaning
def clean_value(value):

    # Remove unnecessary spaces around strings
    if isinstance(value, str):
        return value.strip()

    # Clean each item in a list
    elif isinstance(value, list):
        return [clean_value(item) for item in value]

    # Clean values inside dictionaries
    elif isinstance(value, dict):
        return {
            key: clean_value(item)
            for key, item in value.items()
        }

    # Keep numbers, booleans and other values unchanged
    return value


# Apply cleaning to every destination
cleaned_places = [
    clean_value(place)
    for place in cleaned_places
]

# Update the cleaned dataset
cleaned_data["places"] = cleaned_places


# Check the results
print("Cleaning completed!")
print("Original destinations:", len(places))
print("Cleaned destinations:", len(cleaned_places))

# Confirm the original dataset was not modified
print(
    "Original dataset preserved:",
    tourism_data["places"][0]["name"] == places[0]["name"]
)

In [ ]:
# Check required fields
required_fields = [
    "id",
    "name",
    "type",
    "region",
    "description",
    "activities"
]

validation_errors = []

for index, place in enumerate(cleaned_places):

    for field in required_fields:

        if field not in place or not place[field]:
            validation_errors.append(
                f"Record {index}: Missing {field}"
            )

# Display validation results
if not validation_errors:
    print("Validation successful!")
    print("All required fields are present.")
else:
    print("Validation errors:")
    for error in validation_errors:
        print(error)

print("Total records:", len(cleaned_places))

In [ ]:
cleaned_file = "rwanda_tourism_cleaned.json"

with open(cleaned_file, "w", encoding="utf-8") as file:
    json.dump(
        cleaned_data,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Cleaned dataset saved as:", cleaned_file)

### 3.2 Preparing text documents for Vector RAG

Our tourism dataset is stored as structured JSON.

We will transform each destination into a readable text document.

Each document will contain:
- Destination name
- Location
- Description
- Activities
- Interests
- Best season
- Budget level
- Typical duration
- Costs and safety information
- Travel tips

We will also create metadata for filtering and identifying documents later in ChromaDB.

In [ ]:
# Create an empty list to store our documents
documents = []

# Process each tourism destination
for place in cleaned_places:

    # Build a readable text document
    text = f"""
Destination: {place['name']}

Type: {place['type']}

Location:
Region: {place['region']}
District: {place['district']}

Description:
{place['description']}

Activities:
{', '.join(place['activities'])}

Visitor Interests:
{', '.join(place['interests'])}

Best Season:
{place['best_season']}

Budget Level:
{place['budget_level']}

Typical Duration:
{place['typical_duration']}

Costs Information:
{place['costs_note']}

Safety Information:
{place['safety_note']}

Access from Kigali:
{place['access_from_kigali']}

Travel Tips:
{'; '.join(place['tips']) if place['tips'] else 'No additional tips recorded.'}
""".strip()

    # Create metadata for the document
    metadata = {
        "id": place["id"],
        "name": place["name"],
        "type": place["type"],
        "region": place["region"],
        "district": place["district"],
        "budget_level": place["budget_level"],
        "source_confidence": place["source_confidence"],
        "last_verified": place["last_verified"]
    }

    # Store the document and its metadata together
    documents.append({
        "text": text,
        "metadata": metadata
    })

print("Document preparation completed!")
print("Total documents:", len(documents))

In [ ]:
# Get the first prepared document
first_document = documents[0]

# Display its text
print("DOCUMENT TEXT")
print("=" * 60)
print(first_document["text"])

# Display its metadata
print("\nDOCUMENT METADATA")
print("=" * 60)
print(first_document["metadata"])

In [ ]:
# Check document count
print("Original destinations:", len(cleaned_places))
print("Prepared documents:", len(documents))

# Check for empty documents
empty_documents = [
    doc for doc in documents
    if not doc["text"].strip()
]

print("Empty documents:", len(empty_documents))

# Check for duplicate document IDs
document_ids = [
    doc["metadata"]["id"]
    for doc in documents
]

print("Unique document IDs:", len(set(document_ids)))

# Final result
if (
    len(documents) == len(cleaned_places)
    and len(empty_documents) == 0
    and len(set(document_ids)) == len(documents)
):
    print("\nAll document checks passed!")
else:
    print("\nSome checks failed. We need to investigate.")

### 3.3 Document Length Analysis

Before chunking, we will measure the length of our prepared documents.

This helps us choose an appropriate chunk size rather than splitting documents unnecessarily.

In [ ]:
# Calculate the character length of every document
document_lengths = [
    len(doc["text"])
    for doc in documents
]

print("Total documents:", len(document_lengths))
print("Shortest document:", min(document_lengths), "characters")
print("Longest document:", max(document_lengths), "characters")
print(
    "Average document length:",
    round(sum(document_lengths) / len(document_lengths), 2),
    "characters"
)

### 3.4 Creating document chunks

Our documents are relatively short, so we will use one complete destination document as one chunk.

This preserves the context of each destination and avoids unnecessary splitting.

If we later introduce longer documents, we can apply recursive text splitting.

In [ ]:
# Keep each complete destination document as one chunk
chunks = documents.copy()

# Create a list of chunk IDs
chunk_ids = [
    chunk["metadata"]["id"]
    for chunk in chunks
]

# Display the results
print("Total chunks:", len(chunks))
print("Unique chunk IDs:", len(set(chunk_ids)))

# Inspect one chunk
print("\nExample chunk:")
print(chunks[0]["text"])

### 4.1 Installing the embedding library

We will use Sentence Transformers to load the BGE-M3 model.

This library allows us to convert our tourism documents into numerical vectors.

In [ ]:
%pip install -q sentence-transformers

### 4.2 Loading BGE-M3

We will load the pretrained BGE-M3 model.

Since our Colab environment has a Tesla T4 GPU, we will use CUDA to accelerate embedding generation.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

# Select the available computing device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the pretrained BGE-M3 model
embedding_model = SentenceTransformer(
    "BAAI/bge-m3",
    device=device
)

print("Embedding model loaded successfully!")
print("Model: BAAI/bge-m3")
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "Not available")

In [ ]:
# Extract the text from our chunks
texts = [chunk["text"] for chunk in chunks]

# Generate dense embeddings
embeddings = embedding_model.encode(
    texts,
    batch_size=4,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding generation completed!")
print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape)

In [ ]:
# Display the first destination's embedding
print("First destination:", chunks[0]["metadata"]["name"])

print("\nFirst 10 embedding values:")
print(embeddings[0][:10])

print("\nEmbedding shape:", embeddings[0].shape)

In [ ]:
# Display the first destination's embedding
print("First destination:", chunks[0]["metadata"]["name"])

print("\nFirst 10 embedding values:")
print(embeddings[0][:10])

print("\nEmbedding shape:", embeddings[0].shape)

### 4.3 Testing Semantic Similarity

We will test whether BGE-M3 can identify destinations based on the meaning of a question.

We will:
1. Convert a visitor's question into an embedding.
2. Compare it with our 19 destination embeddings.
3. Calculate cosine similarity.
4. Retrieve the three most similar destinations.

A higher similarity score indicates a closer match in the embedding space.

In [ ]:
import numpy as np

def semantic_search(query, top_k=3):

    # Convert the question into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # Calculate cosine similarity with all destination embeddings
    similarity_scores = embeddings @ query_embedding

    # Get the indices of the highest scores
    best_indices = np.argsort(similarity_scores)[::-1][:top_k]

    # Display the results
    print("Visitor question:", query)
    print("=" * 60)

    for rank, index in enumerate(best_indices, start=1):

        print(f"\nRank {rank}")
        print("Destination:", chunks[index]["metadata"]["name"])
        print("Similarity score:", round(float(similarity_scores[index]), 4))
        print("Description:", chunks[index]["text"].split("Description:")[1].split("Activities:")[0].strip())


# Test our semantic search
semantic_search(
    "Where can I go to see mountain gorillas in Rwanda?"
)

In [ ]:
semantic_search(
    "I want to go hiking and enjoy beautiful natural scenery."
)

semantic_search(
    "Where can I experience Rwandan culture and traditions?"
)

semantic_search(
    "Which destinations are suitable for wildlife lovers?"
)

## 5. ChromaDB – Vector Database Implementation

In this section, we will integrate ChromaDB into our Rwanda Tourism Assistant.

ChromaDB will store our tourism document chunks, their metadata and their corresponding BGE-M3 embeddings.

This will allow us to perform semantic similarity searches and retrieve relevant tourism information when visitors ask questions.

We will:
- Install and configure ChromaDB.
- Create a vector collection.
- Store our document chunks and embeddings.
- Retrieve relevant documents using semantic search.
- Test metadata filtering.
- Verify the retrieval results.

In [ ]:
%pip install -q chromadb

In [ ]:
# Import ChromaDB
import chromadb

# Confirm successful import
print("ChromaDB imported successfully!")
print("ChromaDB version:", chromadb.__version__)

In [ ]:
# Initialize a persistent ChromaDB client
client = chromadb.PersistentClient(
    path="./rwanda_tourism_db"
)

# Confirm successful initialization
print("ChromaDB client initialized successfully!")
print("Database path: ./rwanda_tourism_db")

### 5.1 Creating a ChromaDB Collection

In this section, we will create a collection in ChromaDB to organize and store our Rwanda tourism documents and their corresponding embeddings.

A collection in ChromaDB works like a table in a traditional database. It groups related documents, embeddings, unique identifiers and metadata together.

Our collection will be named `rwanda_tourism`.

We will configure the collection to use cosine similarity when comparing embeddings.

**Why cosine similarity?**

Our BGE-M3 embeddings are normalized. Cosine similarity allows us to measure how semantically similar two vectors are based on their direction.

This will help us retrieve tourism destinations that are most relevant to visitors' questions.

In [ ]:
# Create a collection for Rwanda tourism documents
collection = client.get_or_create_collection(
    name="rwanda_tourism",
    metadata={
        "hnsw:space": "cosine"
    }
)

# Confirm successful collection creation
print("Collection created successfully!")
print("Collection name:", collection.name)

### 5.2 Preparing Data for ChromaDB Insertion

Before inserting our tourism information into ChromaDB, we need to organize the three main components required for storage.

1. **Documents:** The textual information describing each Rwanda tourism destination.
2. **Embeddings:** The numerical vector representations generated using our BGE-M3 model.
3. **Metadata:** Additional information describing each destination, such as its name, region, destination type and budget level.

We have already prepared these components in our previous sections.

In this step, we will extract them into separate lists and ensure that they are correctly aligned.

Each document must have its corresponding embedding, metadata and unique identifier.

This organization will allow ChromaDB to store and retrieve the correct tourism information.

In [ ]:
# Extract tourism document texts
chroma_documents = [
    chunk["text"] for chunk in chunks
]

# Extract document metadata
chroma_metadatas = [
    chunk["metadata"] for chunk in chunks
]

# Extract unique document IDs
chroma_ids = [
    chunk["metadata"]["id"] for chunk in chunks
]

# Confirm the number of records
print("DATA PREPARATION SUMMARY")
print("-" * 40)

print("Documents:", len(chroma_documents))
print("Embeddings:", len(embeddings))
print("Metadata:", len(chroma_metadatas))
print("Unique IDs:", len(chroma_ids))

# Verify that all components have matching lengths
assert len(chroma_documents) == len(embeddings)
assert len(chroma_documents) == len(chroma_metadatas)
assert len(chroma_documents) == len(chroma_ids)

print("\nAll components are aligned successfully!")

### 5.3 Inserting Tourism Documents into ChromaDB

Now that our documents, embeddings, metadata and unique identifiers are prepared, we can insert them into our ChromaDB collection.

ChromaDB provides the `add()` method to store these components together.

During insertion:
- `ids` identify each tourism document uniquely.
- `documents` contain the textual tourism information.
- `embeddings` contain the BGE-M3 numerical representations.
- `metadatas` contain additional destination information.

We will insert all 19 tourism destinations into our `rwanda_tourism` collection.

We will provide the embeddings directly because we have already generated them using BGE-M3.

This avoids generating embeddings again and ensures that ChromaDB stores the same vectors we previously tested.

In [ ]:
# Insert tourism documents into ChromaDB
collection.add(
    ids=chroma_ids,
    documents=chroma_documents,
    embeddings=embeddings.tolist(),
    metadatas=chroma_metadatas
)

# Confirm successful insertion
print("Tourism documents inserted successfully!")
print("Total records stored:", collection.count())

### 5.4 Verifying Stored Documents in ChromaDB

After inserting our tourism documents, we need to verify that the data has been stored correctly.

A successful insertion message does not necessarily mean that all records contain the information we expect.

We will use ChromaDB's `get()` method to retrieve stored records and inspect their contents.

We will:
- Check the total number of stored documents.
- Retrieve a sample of tourism records.
- Inspect their document IDs.
- Examine the stored document text and metadata.

This verification helps us confirm that our vector database contains the expected tourism information before implementing semantic retrieval.

In [ ]:
# Retrieve the first five tourism records
stored_data = collection.get(
    limit=5,
    include=["documents", "metadatas"]
)

# Display the total number of records
print("TOTAL RECORDS IN CHROMADB:", collection.count())

# Inspect the retrieved records
print("\nSTORED TOURISM RECORDS")
print("=" * 60)

for i in range(len(stored_data["ids"])):

    print(f"\nRecord {i + 1}")
    print("-" * 40)

    print("ID:", stored_data["ids"][i])
    print("Name:", stored_data["metadatas"][i]["name"])
    print("Region:", stored_data["metadatas"][i]["region"])

    print("\nDocument preview:")
    print(stored_data["documents"][i][:300])

### 5.5 Implementing Semantic Search with ChromaDB

Now that our tourism documents and embeddings are stored in ChromaDB, we can implement semantic search.

Semantic search allows our system to retrieve information based on the meaning and context of a visitor's question rather than relying only on exact keyword matching.

For example, a visitor might ask:

"Where can I see mountain gorillas?"

Our system should understand the meaning of this question and retrieve Volcanoes National Park because it contains information about mountain gorilla trekking.

**How semantic search works:**

1. A visitor submits a question.
2. BGE-M3 converts the question into a numerical embedding.
3. ChromaDB compares the question embedding with the stored document embeddings.
4. ChromaDB retrieves the most relevant tourism documents.
5. The retrieved documents are returned with their metadata and distance scores.

We will use ChromaDB's `query()` method to perform similarity searches.

Unlike `get()`, which retrieves records directly, `query()` searches for documents based on vector similarity.

Our search will use cosine distance, where a smaller distance indicates a closer match.

In [ ]:
# Define a semantic search function using ChromaDB

def chroma_semantic_search(query, top_k=3):

    # Step 1: Convert the visitor's question into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Step 2: Search ChromaDB for the most relevant documents
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    # Step 3: Display the search results
    print("Visitor Question:", query)
    print("=" * 60)

    for rank in range(len(results["ids"][0])):

        print(f"\nResult {rank + 1}")
        print("-" * 40)

        print("Destination:",
              results["metadatas"][0][rank]["name"])

        print("Region:",
              results["metadatas"][0][rank]["region"])

        print("Cosine Distance:",
              round(results["distances"][0][rank], 4))

        print("\nDocument Preview:")
        print(results["documents"][0][rank][:300])


# Test our semantic search function
chroma_semantic_search(
    "Where can I go to see lakes in Tanzania?"
)

### 5.6 Testing the Semantic Search Function

In this section, we test the semantic search function using different questions that a tourist might ask about destinations in Rwanda.

The objective is to evaluate whether ChromaDB can retrieve relevant destinations based on the meaning and context of a visitor's question rather than relying only on exact keyword matching.

We will use different types of tourism questions, including questions about wildlife, lakes, hiking, and cultural experiences.

The retrieved results will help us understand the strengths and limitations of our current vector retrieval system.

In [ ]:
# Define sample questions that tourists might ask
test_queries = [
    "Where can I see wildlife in Rwanda?",
    "Which places are good for hiking and nature walks?",
    "Where can I enjoy lakes and water activities?",
    "Where can I experience Rwandan culture?",
    "Which destinations are suitable for bird watching?"
]

# Test semantic search with each question
for query in test_queries:
    print("\n" + "=" * 70)
    chroma_semantic_search(query, top_k=3)

### 5.7 Understanding ChromaDB Retrieval Limitations

Although semantic search retrieves relevant destinations based on meaning, it has some limitations.

- **Ranking limitations:** The most semantically similar result is not always the most relevant destination for the visitor's specific request.
- **Broad retrieval:** The search currently considers all destination types, including national parks, lakes, museums, and memorials.
- **Context limitations:** Similarity scores measure semantic distance, but they do not guarantee that the retrieved information fully answers the visitor's question.

These limitations demonstrate why we need additional retrieval techniques, such as metadata filtering and knowledge graph retrieval.

In the next sections, we will improve the retrieval process while maintaining the semantic search functionality.

## 6. Metadata-Based Retrieval

ChromaDB supports metadata filtering, allowing us to narrow down search results using structured information stored alongside documents.

In our Rwanda tourism assistant, each destination contains metadata such as its name, type, region, district, budget level, and best visiting season.

Metadata filtering will help us retrieve destinations that satisfy specific conditions before applying semantic similarity search.

For example, when a visitor asks about lakes in Rwanda, we can filter destinations by their type (`lake`) and then use semantic search to identify the most relevant results.

This approach combines structured filtering with semantic retrieval to improve the relevance of search results.

In [ ]:
# Retrieve metadata for the first five destinations
metadata_results = collection.get(
    limit=5,
    include=["metadatas"]
)

# Display the metadata for each destination
for index, metadata in enumerate(metadata_results["metadatas"], start=1):
    print(f"\nDestination {index}")
    print("-" * 40)

    for key, value in metadata.items():
        print(f"{key}: {value}")

### 6.3 Implementing Metadata Filtering

In this section, we implement metadata filtering in ChromaDB.

Metadata filtering allows us to restrict the search to documents that satisfy specific conditions.

For example, if a visitor asks about lakes, we can use the `type` metadata field to retrieve only destinations categorized as lakes.

We will combine this filter with semantic search so that ChromaDB first considers destinations matching the specified category and then ranks them according to their semantic similarity to the visitor's question.

In [ ]:
# Define the visitor's question
query = "Where can I enjoy lakes and water activities in Rwanda?"

# Generate the query embedding using BGE-M3
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Perform semantic search with a metadata filter
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3,
    where={"type": "lake"},
    include=["documents", "metadatas", "distances"]
)

# Display the filtered search results
print("Visitor Question:", query)
print("=" * 60)

for rank in range(len(results["ids"][0])):
    metadata = results["metadatas"][0][rank]

    print(f"\nResult {rank + 1}")
    print("-" * 40)
    print("Destination:", metadata["name"])
    print("Type:", metadata["type"])
    print("Region:", metadata["region"])
    print("Cosine Distance:", round(results["distances"][0][rank], 4))

### 6.4 Testing Different Metadata Filters

In this section, we test metadata filtering using different destination attributes.

We will explore how to retrieve destinations based on their category, geographical region, and budget level.

This will help us understand how structured metadata can support more specific tourism queries and complement semantic search.

In [ ]:
# Define different metadata filters
test_filters = [
    {"type": "national_park"},
    {"region": "Western Province"},
    {"budget_level": "mid"}
]

# Test each metadata filter
for metadata_filter in test_filters:

    print("\n" + "=" * 60)
    print("Metadata Filter:", metadata_filter)
    print("=" * 60)

    filtered_results = collection.get(
        where=metadata_filter,
        include=["metadatas"]
    )

    # Display matching destinations
    for metadata in filtered_results["metadatas"]:
        print(
            f"Destination: {metadata['name']} | "
            f"Type: {metadata['type']} | "
            f"Region: {metadata['region']} | "
            f"Budget: {metadata['budget_level']}"
        )

    print("Total matches:", len(filtered_results["ids"]))

### 6.5 Combining Multiple Metadata Conditions

ChromaDB supports combining multiple metadata conditions using logical operators.

In this section, we use the `$and` operator to retrieve destinations that satisfy all specified conditions.

For example, we can search for destinations that belong to the lake category and have a mid-level budget.

Combining metadata conditions allows our tourism assistant to support more specific visitor preferences and improve retrieval precision.

In [ ]:
# Define multiple metadata conditions
metadata_filter = {
    "$and": [
        {"type": "lake"},
        {"budget_level": "mid"}
    ]
}

# Retrieve destinations matching both conditions
filtered_results = collection.get(
    where=metadata_filter,
    include=["metadatas"]
)

# Display matching destinations
print("Destinations matching both conditions")
print("=" * 60)

for metadata in filtered_results["metadatas"]:
    print(f"Destination: {metadata['name']}")
    print(f"Type: {metadata['type']}")
    print(f"Region: {metadata['region']}")
    print(f"Budget: {metadata['budget_level']}")
    print("-" * 40)

print("Total matches:", len(filtered_results["ids"]))

### 6.6 Combining Semantic Search with Multiple Metadata Filters

In this section, we combine semantic search with multiple metadata conditions.

Unlike metadata-only retrieval, this approach first restricts the search to destinations satisfying the specified conditions and then ranks those destinations according to their semantic similarity to the visitor's question.

For example, a visitor may ask for affordable lake destinations where they can enjoy water activities.

We can filter destinations by type and budget level, then use semantic search to identify the most relevant results.

This combination improves retrieval precision by considering both visitor preferences and the meaning of their questions.

In [ ]:
# Define the visitor's question
query = "Where can I enjoy water activities and relaxing lakeside experiences?"

# Generate the question embedding
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Define multiple metadata conditions
metadata_filter = {
    "$and": [
        {"type": "lake"},
        {"budget_level": "mid"}
    ]
}

# Perform semantic search with metadata filtering
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3,
    where=metadata_filter,
    include=["documents", "metadatas", "distances"]
)

# Display the search results
print("Visitor Question:", query)
print("=" * 60)

for rank in range(len(results["ids"][0])):
    metadata = results["metadatas"][0][rank]

    print(f"\nResult {rank + 1}")
    print("-" * 40)
    print("Destination:", metadata["name"])
    print("Type:", metadata["type"])
    print("Region:", metadata["region"])
    print("Budget:", metadata["budget_level"])
    print("Cosine Distance:", round(results["distances"][0][rank], 4))

### 6.7 Creating a Reusable Hybrid Retrieval Function

To make our retrieval system reusable, we will create a function that combines semantic search and metadata filtering.

The function will accept a visitor's question, the number of results to retrieve, and optional metadata conditions.

When metadata conditions are provided, ChromaDB will filter the documents before ranking them using semantic similarity.

When no metadata conditions are provided, the function will perform ordinary semantic search across the entire collection.

This reusable function will later serve as one of the retrieval tools available to our LangGraph agent.

In [ ]:
# Define a reusable hybrid retrieval function
def hybrid_retrieval(query, top_k=3, metadata_filter=None):

    # Generate the embedding for the visitor's question
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Prepare the ChromaDB search parameters
    search_params = {
        "query_embeddings": query_embedding.tolist(),
        "n_results": top_k,
        "include": ["documents", "metadatas", "distances"]
    }

    # Add metadata filtering only when provided
    if metadata_filter is not None:
        search_params["where"] = metadata_filter

    # Execute the search
    results = collection.query(**search_params)

    return results

### 6.8 Testing the Hybrid Retrieval Function

In this section, we test our reusable hybrid retrieval function with different visitor questions.

We will perform two types of searches:

1. **Unfiltered semantic search:** Retrieves relevant destinations from the entire collection.
2. **Filtered semantic search:** Retrieves relevant destinations that satisfy specified metadata conditions.

The objective is to verify that our function supports both retrieval approaches using a single implementation.

In [ ]:
# Test 1: Semantic search without metadata filtering
query_1 = "Where can I see wildlife in Rwanda?"

results_1 = hybrid_retrieval(
    query=query_1,
    top_k=3
)

print("TEST 1: Unfiltered Semantic Search")
print("=" * 60)

for rank, metadata in enumerate(results_1["metadatas"][0], start=1):
    print(f"{rank}. {metadata['name']} | Type: {metadata['type']}")


# Test 2: Semantic search with metadata filtering
query_2 = "Where can I see wildlife in Rwanda?"

results_2 = hybrid_retrieval(
    query=query_2,
    top_k=3,
    metadata_filter={"type": "national_park"}
)

print("\nTEST 2: Filtered Semantic Search")
print("=" * 60)

for rank, metadata in enumerate(results_2["metadatas"][0], start=1):
    print(f"{rank}. {metadata['name']} | Type: {metadata['type']}")

### 7.1 Installing Neo4j Python Driver

The Neo4j Python Driver allows our Python application to connect
to Neo4j AuraDB, execute Cypher queries, and retrieve graph data.

We will use this driver to create and manage our Rwanda Tourism
Knowledge Graph.

In [ ]:
# Install the Neo4j Python driver
!pip install neo4j

In [ ]:
# Import the Neo4j database driver
from neo4j import GraphDatabase

# Import Google Colab userdata to access stored secrets
from google.colab import userdata

# Load Neo4j connection credentials securely
NEO4J_URI = userdata.get("NEO4J_URI")
NEO4J_USERNAME = userdata.get("NEO4J_USERNAME")
NEO4J_PASSWORD = userdata.get("NEO4J_PASSWORD")

# Verify that the credentials have been loaded
print("Neo4j URI:", NEO4J_URI)
print("Neo4j Username:", NEO4J_USERNAME)
print("Neo4j Password:", "Loaded securely" if NEO4J_PASSWORD else "Missing")

### 7.3 Testing Neo4j Database Connection

In this section, we establish a connection between Google Colab
and our Neo4j AuraDB instance using the Neo4j Python driver.

We use verify_connectivity() to confirm that the database is
reachable and that the supplied credentials are valid.

In [ ]:
# Create a connection to Neo4j AuraDB
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

# Verify the database connection
try:
    driver.verify_connectivity()
    print("Successfully connected to Neo4j AuraDB!")

except Exception as e:
    print("Connection failed:", e)

### 7.4 Verifying the Initial Graph Database

Before creating our tourism knowledge graph, we inspect the Neo4j
database to understand its initial state.

We count the existing nodes and relationships to confirm that
we are starting with a clean database.

In [ ]:
# Define a function to count nodes and relationships
def get_graph_statistics(tx):
    result = tx.run("""
        OPTIONAL MATCH (n)
        WITH count(n) AS node_count

        OPTIONAL MATCH ()-[r]->()
        RETURN node_count, count(r) AS relationship_count
    """)

    return result.single()

# Retrieve graph statistics
with driver.session() as session:
    statistics = session.execute_read(get_graph_statistics)

    print("Total Nodes:", statistics["node_count"])
    print("Total Relationships:", statistics["relationship_count"])

### 7.5 Inspecting the Tourism Dataset for Graph Modeling

Before designing the Neo4j knowledge graph, we inspect the structure
of our Rwanda tourism dataset.

We identify the available fields and examine a sample destination
to understand which attributes should become:

- Node properties
- Separate graph nodes
- Relationships between nodes

This ensures that our graph model reflects the actual tourism data.

In [ ]:
# Import the JSON library
import json

# Load the Rwanda tourism dataset
with open("rwanda_places.json", "r", encoding="utf-8") as file:
    tourism_data = json.load(file)

# Extract the destination records
places = tourism_data["places"]

# Display dataset information
print("Total destinations:", len(places))

# Display available fields
print("\nAvailable fields:")
for field in places[0].keys():
    print("-", field)

# Display one destination as an example
print("\nSample destination:")
print(json.dumps(places[0], indent=4, ensure_ascii=False))

### 7.7 Creating Graph Database Constraints

Before inserting tourism destinations into Neo4j, we create a uniqueness
constraint on the Destination node.

The constraint ensures that every destination has a unique ID and
prevents duplicate destination records from being created.

In [ ]:
# Create a uniqueness constraint for destination IDs
with driver.session() as session:
    session.run("""
        CREATE CONSTRAINT destination_id_unique IF NOT EXISTS
        FOR (d:Destination)
        REQUIRE d.id IS UNIQUE
    """)

    print("Destination uniqueness constraint created successfully!")

### 7.8 Verifying Graph Database Constraints

In this section, we inspect the constraints registered in Neo4j.

We verify that the uniqueness constraint on Destination.id
has been successfully created before inserting tourism data.

In [ ]:
# Retrieve all constraints from Neo4j
with driver.session() as session:
    result = session.run("SHOW CONSTRAINTS")

    for record in result:
        print("Constraint name:", record["name"])
        print("Constraint type:", record["type"])
        print("Entity type:", record["entityType"])
        print("Labels or types:", record["labelsOrTypes"])
        print("Properties:", record["properties"])
        print("-" * 40)

### 7.9 Preparing Destination Data for Neo4j

In this section, we prepare the Rwanda tourism destination records
for insertion into Neo4j.

We extract the main destination properties from the existing JSON
dataset while preserving the original information.

The activities, interests, and moods will be handled separately
when creating graph relationships.

In [ ]:
# Define the destination properties to store in Neo4j
destination_properties = [
    "id",
    "name",
    "type",
    "region",
    "district",
    "best_season",
    "budget_level",
    "typical_duration",
    "description",
    "costs_note",
    "safety_note",
    "access_from_kigali",
    "tips",
    "source_urls",
    "source_confidence",
    "last_verified"
]

# Prepare destination records
prepared_destinations = []

for place in places:
    destination = {
        key: place.get(key)
        for key in destination_properties
    }

    prepared_destinations.append(destination)

# Display preparation results
print("Total prepared destinations:", len(prepared_destinations))

print("\nFirst prepared destination:")
print(json.dumps(
    prepared_destinations[0],
    indent=4,
    ensure_ascii=False
))

### 7.10 Validating Prepared Destination Data

Before inserting destination records into Neo4j, we perform basic
data validation.

We check:
- The total number of prepared destinations.
- Whether every destination has an ID and name.
- Whether destination IDs are unique.
- Whether any required values are missing.

This helps maintain data consistency in our knowledge graph.

In [ ]:
# Check the total number of destinations
total_destinations = len(prepared_destinations)

# Check for missing destination IDs
missing_ids = [
    place["name"]
    for place in prepared_destinations
    if not place["id"]
]

# Check for missing destination names
missing_names = [
    place["id"]
    for place in prepared_destinations
    if not place["name"]
]

# Check for duplicate destination IDs
destination_ids = [
    place["id"]
    for place in prepared_destinations
]

duplicate_ids = [
    destination_id
    for destination_id in set(destination_ids)
    if destination_ids.count(destination_id) > 1
]

# Display validation results
print("Total destinations:", total_destinations)
print("Missing IDs:", len(missing_ids))
print("Missing names:", len(missing_names))
print("Duplicate IDs:", len(duplicate_ids))

# Display validation status
if not missing_ids and not missing_names and not duplicate_ids:
    print("\nData validation successful!")
else:
    print("\nData validation found issues.")

### 7.11 Inserting Tourism Destinations into Neo4j

In this section, we insert the 19 prepared tourism destinations
into our Neo4j graph database.

We use the UNWIND clause to process multiple destination records
and MERGE to identify destinations using their unique IDs.

Each destination is created as a node with the Destination label
and its corresponding properties.

In [ ]:
# Define the Cypher query for inserting destinations
create_destinations_query = """
UNWIND $destinations AS destination

MERGE (d:Destination {id: destination.id})

SET d += destination

RETURN count(d) AS total_inserted
"""

# Insert destination records into Neo4j
with driver.session() as session:
    result = session.run(
        create_destinations_query,
        destinations=prepared_destinations
    )

    total_inserted = result.single()["total_inserted"]

    print("Destinations processed:", total_inserted)

### 7.12 Verifying Destination Nodes in Neo4j

After inserting the tourism destinations, we verify that the
records have been successfully stored in Neo4j.

We retrieve all nodes with the Destination label and inspect
their IDs, names, and types.

This confirms that our tourism destination nodes are available
in the graph database.

In [ ]:
# Retrieve all tourism destinations from Neo4j
with driver.session() as session:
    result = session.run("""
        MATCH (d:Destination)
        RETURN d.id AS id,
               d.name AS name,
               d.type AS type
        ORDER BY d.name
    """)

    destinations = [record.data() for record in result]

# Display destination records
print("Total destinations in Neo4j:", len(destinations))

for destination in destinations:
    print(
        f"{destination['name']} | "
        f"ID: {destination['id']} | "
        f"Type: {destination['type']}"
    )

### 7.13 Preparing Geographical Relationships

In this section, we prepare geographical relationships between
tourism destinations, districts, and regions.

We extract the unique district and region values from our
prepared destination records.

These geographical entities will become separate nodes in Neo4j,
allowing destinations to be connected through their locations.

In [ ]:
# Extract unique districts and regions from destination records
districts = sorted({
    place["district"]
    for place in prepared_destinations
    if place["district"]
})

regions = sorted({
    place["region"]
    for place in prepared_destinations
    if place["region"]
})

# Display geographical entities
print("Total unique districts:", len(districts))
print("Districts:", districts)

print("\nTotal unique regions:", len(regions))
print("Regions:", regions)

### 7.14 Identifying Multiple Geographical Values

Some tourism destinations may cover more than one district or region.

In this section, we identify destination records whose district
or region fields contain multiple geographical values.

This helps us prepare accurate geographical relationships
without treating multiple locations as a single district or region.

In [ ]:
# Identify destinations with multiple district values
print("Destinations with multiple districts:\n")

for place in prepared_destinations:
    if "," in place["district"]:
        print(
            place["name"],
            "->",
            place["district"]
        )

# Identify destinations with multiple region values
print("\nDestinations with multiple regions:\n")

for place in prepared_destinations:
    if "/" in place["region"]:
        print(
            place["name"],
            "->",
            place["region"]
        )

### 7.15 Preparing Geographical Entities

In this section, we normalize the geographical information
from our tourism dataset.

Some destinations cover multiple districts or regions.
We split these values into individual geographical entities.

We prepare unique district and region lists that will later
be represented as separate nodes in Neo4j.

In [ ]:
# Prepare unique district and region entities

district_entities = set()
region_entities = set()

for place in prepared_destinations:

    # Split multiple districts separated by commas
    place_districts = place["district"].split(",")

    for district in place_districts:
        district_entities.add(district.strip())

    # Split multiple regions separated by slashes
    place_regions = place["region"].split("/")

    for region in place_regions:
        region_entities.add(region.strip())

# Convert sets into sorted lists
district_entities = sorted(district_entities)
region_entities = sorted(region_entities)

# Display the normalized geographical entities
print("Total unique districts:", len(district_entities))
print("Districts:", district_entities)

print("\nTotal unique regions:", len(region_entities))
print("Regions:", region_entities)

### 7.16 Standardizing Region Names

The tourism dataset contains inconsistent region naming conventions.

For example, "Eastern" and "Eastern Province" refer to the same
region in our dataset.

We standardize these values to ensure that each geographical
region is represented by one unique node in Neo4j.

Kigali remains unchanged.

In [ ]:
# Define region name standardization rules
region_mapping = {
    "Eastern": "Eastern Province",
    "Southern": "Southern Province",
    "Western": "Western Province"
}

# Apply standardization to region entities
standardized_regions = [
    region_mapping.get(region, region)
    for region in region_entities
]

# Remove duplicates and sort region names
standardized_regions = sorted(set(standardized_regions))

# Display standardized regions
print("Total standardized regions:", len(standardized_regions))
print("Regions:", standardized_regions)

### 7.17 Create Region and District Nodes

In this step, we will create the geographical entities in Neo4j.

We will create two node types:
- `Region`: represents the five standardized regions.
- `District`: represents the 18 unique districts identified in our dataset.

These nodes will later be connected to their corresponding destinations through relationships.

### 7.18 Prepare Destination–Geography Mapping

In this step, we extract each destination's ID, name, district, and region.

We will use this mapping to create accurate geographical relationships in Neo4j, including destinations associated with multiple districts or regions.

In [ ]:
for place in prepared_destinations:
    print({
        "id": place["id"],
        "name": place["name"],
        "districts": [
            d.strip() for d in place["district"].split(",")
        ],
        "regions": [
            r.strip() for r in place["region"].split("/")
        ]
    })

### 7.19 Create Destination–District Relationships

In this step, we connect each Destination node to its corresponding District nodes in Neo4j.

We use the `LOCATED_IN` relationship.

Some destinations cover multiple districts, so one destination may have several `LOCATED_IN` relationships.

### 7.20 Create Destination–Region Relationships

In this step, we connect each Destination node to its corresponding Region nodes.

We use the `IN_REGION` relationship.

Some destinations span multiple regions, so one destination may have multiple `IN_REGION` relationships.

Region names are standardized to match the Region nodes in Neo4j.


### 7.21 Prepare Activity, Interest, and Mood Entities

In this step, we extract unique activities, interests, and moods from all destinations.

These entities will become separate nodes in Neo4j and will help the tourism assistant retrieve destinations based on what visitors want to experience.

In [ ]:
unique_activities = sorted({
    activity
    for place in places
    for activity in place.get("activities", [])
})

unique_interests = sorted({
    interest
    for place in places
    for interest in place.get("interests", [])
})

unique_moods = sorted({
    mood
    for place in places
    for mood in place.get("moods", [])
})

print("Activities:", len(unique_activities))
print(unique_activities)

print("\nInterests:", len(unique_interests))
print(unique_interests)

print("\nMoods:", len(unique_moods))
print(unique_moods)

### 7.22 Create Activity, Interest, and Mood Nodes

In this step, we create three additional node types in Neo4j:

- Activity: represents activities available at destinations.
- Interest: represents visitor interests that destinations match.
- Mood: represents the travel moods associated with destinations.

Each unique value from the dataset becomes a separate node. We use MERGE to prevent duplicate nodes.

### 7.23 Create Destination–Activity Relationships

In this step, we connect each Destination node to its corresponding Activity nodes.

The OFFERS relationship represents activities available at a destination.

This allows the tourism assistant to identify destinations based on activities visitors want to experience.

In [ ]:
for place in places:
    print({
        "id": place["id"],
        "name": place["name"],
        "activities": place.get("activities", [])
    })

### 7.23 Create Destination–Activity Relationships

In this step, we connect each Destination node to its corresponding Activity nodes in Neo4j.

We use the OFFERS relationship to represent activities available at each destination.

A destination can offer multiple activities, and the same activity can be offered by multiple destinations.

### 7.24 Prepare Destination–Interest Mapping

In this step, we extract the interests associated with each destination.

These interests describe what visitors can explore or experience at a destination.

We will use this mapping to create MATCHES_INTEREST relationships in Neo4j.

In [ ]:
for place in places:
    print({
        "id": place["id"],
        "name": place["name"],
        "interests": place.get("interests", [])
    })

### 7.24 Create Destination–Interest Relationships

In this step, we connect each Destination node to its corresponding Interest nodes in Neo4j.

We use the MATCHES_INTEREST relationship to represent visitor interests associated with each destination.

A destination can match multiple interests, and the same interest can be associated with multiple destinations.

### 7.25 Prepare Destination–Mood Mapping

In this step, we extract the moods associated with each destination.

Moods describe the type of travel experience a visitor is looking for, such as adventure, relaxation, culture, wildlife, or romance.

We will use this mapping to create HAS_MOOD relationships in Neo4j.

In [ ]:
for place in places:
    print({
        "id": place["id"],
        "name": place["name"],
        "moods": place.get("moods", [])
    })

### 7.26 Create Destination–Mood Relationships

In this step, we connect each Destination node to its corresponding Mood nodes in Neo4j.

We use the HAS_MOOD relationship to represent the travel moods associated with each destination.

A destination can match multiple moods, and the same mood can be associated with multiple destinations.

### 7.27 Verify the Complete Knowledge Graph

In this step, we verify the complete Neo4j knowledge graph.

We check the number of nodes for each entity type and the number of relationships for each relationship type.

This verification helps us confirm that the graph has been constructed successfully before integrating it with our Google Colab notebook.

### 7.28 Verify Neo4j Retrieval from Google Colab

In this step, we use the Neo4j Python driver to retrieve information from our completed knowledge graph.

This confirms that Google Colab can communicate with Neo4j AuraDB and access the graph data.

The graph construction remains in Neo4j, while Colab will handle graph retrieval and integration with our RAG system.

In [ ]:
query = """
MATCH (d:Destination)
RETURN d.id AS id, d.name AS destination
ORDER BY d.name
LIMIT 10
"""

with driver.session() as session:
    result = session.run(query)
    destinations = [record.data() for record in result]

for destination in destinations:
    print(destination)

### 7.28 Implement Graph Retrieval Function

In this step, we implement a reusable graph retrieval function using the Neo4j Python driver.

The function accepts a search keyword and retrieves matching destinations based on:
- Destination names
- Activities
- Visitor interests
- Travel moods
- Regions
- Districts

For each matching destination, it returns its associated geographical information and travel characteristics.

This function will later serve as one of the retrieval tools used by our Agentic Hybrid RAG system.

In [ ]:
def neo4j_graph_retrieval(query, top_k=5):

    cypher_query = """
    MATCH (d:Destination)

    OPTIONAL MATCH (d)-[:OFFERS]->(a:Activity)
    OPTIONAL MATCH (d)-[:MATCHES_INTEREST]->(i:Interest)
    OPTIONAL MATCH (d)-[:HAS_MOOD]->(m:Mood)
    OPTIONAL MATCH (d)-[:IN_REGION]->(r:Region)
    OPTIONAL MATCH (d)-[:LOCATED_IN]->(dist:District)

    WITH d,
         collect(DISTINCT a.name) AS activities,
         collect(DISTINCT i.name) AS interests,
         collect(DISTINCT m.name) AS moods,
         collect(DISTINCT r.name) AS regions,
         collect(DISTINCT dist.name) AS districts

    WHERE
        toLower(d.name) CONTAINS toLower($query)
        OR any(x IN activities WHERE toLower(x) CONTAINS toLower($query))
        OR any(x IN interests WHERE toLower(x) CONTAINS toLower($query))
        OR any(x IN moods WHERE toLower(x) CONTAINS toLower($query))
        OR any(x IN regions WHERE toLower(x) CONTAINS toLower($query))
        OR any(x IN districts WHERE toLower(x) CONTAINS toLower($query))

    RETURN
        d.id AS id,
        d.name AS destination,
        d.type AS type,
        regions,
        districts,
        activities,
        interests,
        moods

    ORDER BY destination
    LIMIT $top_k
    """

    with driver.session() as session:
        result = session.run(
            cypher_query,
            query=query,
            top_k=top_k
        )

        return [record.data() for record in result]

### 7.28 Implement Graph Retrieval Function

This function retrieves destinations from Neo4j by searching destination names, activities, interests, moods, regions, and districts.

The search term is passed as a Cypher parameter named `search_term` to avoid conflicts with the Neo4j driver's `session.run()` method.

In [ ]:
def neo4j_graph_retrieval(search_term, top_k=5):

    cypher_query = """
    MATCH (d:Destination)

    OPTIONAL MATCH (d)-[:OFFERS]->(a:Activity)
    OPTIONAL MATCH (d)-[:MATCHES_INTEREST]->(i:Interest)
    OPTIONAL MATCH (d)-[:HAS_MOOD]->(m:Mood)
    OPTIONAL MATCH (d)-[:IN_REGION]->(r:Region)
    OPTIONAL MATCH (d)-[:LOCATED_IN]->(dist:District)

    WITH d,
         collect(DISTINCT a.name) AS activities,
         collect(DISTINCT i.name) AS interests,
         collect(DISTINCT m.name) AS moods,
         collect(DISTINCT r.name) AS regions,
         collect(DISTINCT dist.name) AS districts

    WHERE
        toLower(d.name) CONTAINS toLower($search_term)
        OR any(x IN activities WHERE toLower(x) CONTAINS toLower($search_term))
        OR any(x IN interests WHERE toLower(x) CONTAINS toLower($search_term))
        OR any(x IN moods WHERE toLower(x) CONTAINS toLower($search_term))
        OR any(x IN regions WHERE toLower(x) CONTAINS toLower($search_term))
        OR any(x IN districts WHERE toLower(x) CONTAINS toLower($search_term))

    RETURN
        d.id AS id,
        d.name AS destination,
        d.type AS type,
        regions,
        districts,
        activities,
        interests,
        moods

    ORDER BY destination
    LIMIT $top_k
    """

    with driver.session() as session:
        result = session.run(
            cypher_query,
            search_term=search_term,
            top_k=top_k
        )

        return [record.data() for record in result]

In [ ]:
results = neo4j_graph_retrieval("gorilla", top_k=5)

for item in results:
    print(item)
    print("-" * 80)

### 7.29 Test Different Graph Search Queries

In this step, we evaluate our Neo4j graph retrieval function using different search terms.

We test activities, interests, moods, and geographical terms to confirm that the graph can retrieve relevant destinations through their relationships.

In [ ]:
test_queries = [
    "birdwatching",
    "hiking",
    "relaxing",
    "Kigali",
    "museum"
]

for search_term in test_queries:
    print(f"\nSearch: {search_term}")
    print("-" * 60)

    results = neo4j_graph_retrieval(search_term, top_k=5)

    for item in results:
        print(f"- {item['destination']}")

    if not results:
        print("No matching destinations found.")

### 7.30 Graph Retrieval Limitations

Neo4j graph retrieval provides structured access to destinations and their relationships. However, our current implementation has some limitations:

1. Keyword dependency: It uses exact keyword containment rather than semantic similarity.
2. Vocabulary differences: It may fail to connect related terms with different wording, such as "peaceful holiday" and "relaxing".
3. No ranking by relevance: Results are currently ordered alphabetically rather than ranked by relevance.
4. Limited query understanding: It does not independently interpret complex natural-language questions.
5. Dataset dependency: Retrieval is limited to the entities and relationships available in our knowledge graph.

These limitations motivate combining Neo4j with ChromaDB and using LangGraph to coordinate retrieval strategies.

### 7.31 Comparison of Vector Retrieval and Graph Retrieval

Our project uses two complementary retrieval approaches.

**Vector Retrieval (ChromaDB):**
- Uses embeddings to find semantically similar documents.
- Supports natural-language queries and different wording.
- Retrieves contextual information from destination descriptions.
- May not explicitly represent relationships between entities.

**Graph Retrieval (Neo4j):**
- Uses nodes and relationships to retrieve connected information.
- Supports structured searches across destinations, activities, interests, moods, regions, and districts.
- Provides explicit relationships between tourism entities.
- Our current implementation relies on keyword matching.

**Conclusion:**

Combining both approaches allows the tourism assistant to benefit from semantic similarity and structured relationship-based retrieval.

### 7.32 Prepare for Hybrid Retrieval Integration

In this step, we test ChromaDB and Neo4j independently using the same query.

This allows us to compare their retrieval results before combining them into a hybrid retrieval system.

In [ ]:
test_query = "I want to see gorillas and enjoy nature"

# Vector retrieval
vector_results = hybrid_retrieval(
    query=test_query,
    top_k=3
)

# Graph retrieval
graph_results = neo4j_graph_retrieval(
    search_term="gorilla",
    top_k=3
)

print("VECTOR RETRIEVAL — CHROMADB")
print("=" * 60)

for result in vector_results:
    print(result)

print("\nGRAPH RETRIEVAL — NEO4J")
print("=" * 60)

for result in graph_results:
    print(result)

### 7.33 Inspect ChromaDB Retrieval Results

In this step, we inspect the actual documents, metadata, and similarity distances returned by ChromaDB.

This helps us understand which destinations were retrieved for the query and compare them with Neo4j graph retrieval.

In [ ]:
print("CHROMADB RETRIEVED DESTINATIONS")
print("=" * 60)

for i, document in enumerate(vector_results["documents"][0]):
    print(f"\nResult {i + 1}")
    print("Document:", document)
    print("Metadata:", vector_results["metadatas"][0][i])
    print("Distance:", vector_results["distances"][0][i])

### 7.34 Implement Hybrid Retrieval

In this step, we combine ChromaDB semantic retrieval and Neo4j graph retrieval.

The function will:
1. Search ChromaDB using the complete natural-language query.
2. Search Neo4j using a keyword extracted from the query.
3. Combine the retrieved destinations.
4. Remove duplicate destinations using their unique IDs.

This creates a unified result set from both retrieval systems.

In [ ]:
def hybrid_graph_vector_retrieval(
    query,
    graph_keyword,
    top_k=3
):
    # 1. Retrieve from ChromaDB
    vector_results = hybrid_retrieval(
        query=query,
        top_k=top_k
    )

    # 2. Retrieve from Neo4j
    graph_results = neo4j_graph_retrieval(
        search_term=graph_keyword,
        top_k=top_k
    )

    # 3. Combine results and remove duplicates
    combined_results = {}

    for i, destination_id in enumerate(vector_results["ids"][0]):
        combined_results[destination_id] = {
            "id": destination_id,
            "source": ["ChromaDB"],
            "document": vector_results["documents"][0][i],
            "metadata": vector_results["metadatas"][0][i]
        }

    for result in graph_results:
        destination_id = result["id"]

        if destination_id in combined_results:
            combined_results[destination_id]["source"].append("Neo4j")
            combined_results[destination_id]["graph_data"] = result
        else:
            combined_results[destination_id] = {
                "id": destination_id,
                "source": ["Neo4j"],
                "graph_data": result
            }

    return list(combined_results.values())

### 7.35 Test Hybrid Retrieval

In this step, we test the combined retrieval system using a natural-language query and a graph keyword.

We will inspect:
- Which destinations were retrieved.
- Whether each destination came from ChromaDB, Neo4j, or both.
- Whether duplicate destinations were removed.

In [ ]:
test_query = "I want to see gorillas and enjoy nature"
graph_keyword = "gorilla"

hybrid_results = hybrid_graph_vector_retrieval(
    query=test_query,
    graph_keyword=graph_keyword,
    top_k=3
)

print("HYBRID RETRIEVAL RESULTS")
print("=" * 60)

for result in hybrid_results:
    print(f"\nDestination ID: {result['id']}")
    print(f"Retrieved from: {', '.join(result['source'])}")

    if "metadata" in result:
        print(f"Destination: {result['metadata']['name']}")

    if "graph_data" in result:
        print(f"Graph relationships: {result['graph_data']}")

### 7.36 Test Hybrid Retrieval — Lake and Water Activities

In this test, we evaluate how the hybrid retrieval system handles a query about relaxing near a lake and enjoying water activities.

We compare semantic retrieval from ChromaDB with relationship-based retrieval from Neo4j.

In [ ]:
test_query = "I want to relax near a lake and enjoy water activities"
graph_keyword = "lake"

hybrid_results = hybrid_graph_vector_retrieval(
    query=test_query,
    graph_keyword=graph_keyword,
    top_k=3
)

for result in hybrid_results:
    print(f"\nDestination: {result.get('metadata', {}).get('name', result.get('graph_data', {}).get('destination'))}")
    print(f"Retrieved from: {', '.join(result['source'])}")

### 7.37 Test Hybrid Retrieval — Culture and History

In this final test, we evaluate the hybrid retrieval system using a query about cultural experiences and historical places in Rwanda.

We will observe how ChromaDB and Neo4j complement each other when retrieving destinations related to culture and history.

In [ ]:
test_query = "I want to explore Rwandan culture, traditional heritage and historical places"
graph_keyword = "culture"

hybrid_results = hybrid_graph_vector_retrieval(
    query=test_query,
    graph_keyword=graph_keyword,
    top_k=3
)

for result in hybrid_results:
    destination = (
        result.get("metadata", {}).get("name")
        or result.get("graph_data", {}).get("destination")
    )

    print(f"\nDestination: {destination}")
    print(f"Retrieved from: {', '.join(result['source'])}")

8.Agent Development Using LangGraph

### 8.2 Install LangGraph

LangGraph is a framework for building stateful, multi-step AI agent workflows.

We will use it to coordinate the ChromaDB and Neo4j retrieval tools and later integrate Qwen for answer generation.


In [ ]:
%pip install -q -U langgraph

### 8.3 Verify LangGraph Installation

In this step, we verify that LangGraph has been installed successfully and can be imported into our Google Colab environment.

In [ ]:
from importlib.metadata import version

print("LangGraph version:", version("langgraph"))
print("LangGraph installed successfully!")

### 8.4 Define the Agent State

LangGraph uses a shared state to pass information between different nodes in an agent workflow.

We define the state using Python's TypedDict. Each node can read information from the state and update the fields required by the next node.

In [ ]:
from typing import TypedDict, List, Dict, Any

class TourismAgentState(TypedDict):
    question: str
    vector_results: List[Dict[str, Any]]
    graph_results: List[Dict[str, Any]]
    final_answer: str

### 8.5 Create the ChromaDB Retrieval Node

A LangGraph node is a Python function that performs a specific task in the agent workflow.

This node:
1. Receives the user's question from the shared state.
2. Searches ChromaDB using semantic retrieval.
3. Stores the retrieved documents in `vector_results`.
4. Returns the updated state.

In [ ]:
def chroma_retrieval_node(state: TourismAgentState):

    results = hybrid_retrieval(
        query=state["question"],
        top_k=3
    )

    vector_results = []

    for i, document in enumerate(results["documents"][0]):
        vector_results.append({
            "id": results["ids"][0][i],
            "document": document,
            "metadata": results["metadatas"][0][i]
        })

    return {
        "vector_results": vector_results
    }

### 8.6 Create the Neo4j Retrieval Node

This node connects the LangGraph workflow to our Neo4j knowledge graph.

It:
1. Receives the user's question.
2. Calls the existing `neo4j_graph_retrieval()` function.
3. Retrieves matching destinations and their relationships.
4. Stores the results in `graph_results`.

In [ ]:
def neo4j_retrieval_node(state: TourismAgentState):

    results = neo4j_graph_retrieval(
        search_term=state["question"],
        top_k=3
    )

    return {
        "graph_results": results
    }

### 8.7 Build the LangGraph Workflow

LangGraph uses a StateGraph to define and connect workflow nodes.

In this step:
1. Initialize the StateGraph using TourismAgentState.
2. Register the ChromaDB and Neo4j retrieval nodes.
3. Connect the nodes using edges.
4. Define the workflow entry and ending points.
5. Compile the graph into an executable application.

In [ ]:
from langgraph.graph import StateGraph, START, END

# Initialize the workflow
workflow = StateGraph(TourismAgentState)

# Register retrieval nodes
workflow.add_node("chroma_retrieval", chroma_retrieval_node)
workflow.add_node("neo4j_retrieval", neo4j_retrieval_node)

# Define execution order
workflow.add_edge(START, "chroma_retrieval")
workflow.add_edge("chroma_retrieval", "neo4j_retrieval")
workflow.add_edge("neo4j_retrieval", END)

# Compile workflow
tourism_agent = workflow.compile()

print("LangGraph workflow compiled successfully!")

### 8.8 Visualize the LangGraph Workflow

LangGraph provides a visualization feature that displays the workflow nodes and their execution sequence.

This helps us understand how the ChromaDB and Neo4j retrieval nodes are connected.

In [ ]:
from IPython.display import Image, display

display(
    Image(
        tourism_agent.get_graph().draw_mermaid_png()
    )
)

### 8.9 Test the LangGraph Workflow

In this step, we execute the compiled LangGraph workflow with a tourism question.

We verify that:
- ChromaDB retrieves relevant documents.
- Neo4j retrieves related graph information.
- Both results are returned in the shared agent state.

In [ ]:
initial_state = {
    "question": "I want to see gorillas and enjoy nature",
    "vector_results": [],
    "graph_results": [],
    "final_answer": ""
}

result = tourism_agent.invoke(initial_state)

print("CHROMADB RESULTS")
print("=" * 50)

for item in result["vector_results"]:
    print(item["metadata"]["name"])

print("\nNEO4J RESULTS")
print("=" * 50)

for item in result["graph_results"]:
    print(item["destination"])

### 8.10 Update the Agent State

Neo4j currently requires a searchable keyword rather than a complete natural-language question.

We will add a graph_keyword field to the shared state so the workflow can pass an appropriate keyword to Neo4j.

Later, Qwen will help the agent extract this keyword automatically.

In [ ]:
class TourismAgentState(TypedDict):
    question: str
    graph_keyword: str
    vector_results: List[Dict[str, Any]]
    graph_results: List[Dict[str, Any]]
    final_answer: str

### 8.11 Update the Neo4j Retrieval Node

We update the Neo4j retrieval node to read the graph_keyword field from the shared state.

This allows Neo4j to search using a specific keyword while ChromaDB continues using the complete natural-language question.

In [ ]:
def neo4j_retrieval_node(state: TourismAgentState):

    results = neo4j_graph_retrieval(
        search_term=state["graph_keyword"],
        top_k=3
    )

    return {
        "graph_results": results
    }

### 8.12 Rebuild the LangGraph Workflow

We rebuild the workflow to register the updated Neo4j retrieval node.

The workflow will execute ChromaDB retrieval first, followed by Neo4j retrieval, using the graph keyword supplied in the shared state.

In [ ]:
workflow = StateGraph(TourismAgentState)

workflow.add_node("chroma_retrieval", chroma_retrieval_node)
workflow.add_node("neo4j_retrieval", neo4j_retrieval_node)

workflow.add_edge(START, "chroma_retrieval")
workflow.add_edge("chroma_retrieval", "neo4j_retrieval")
workflow.add_edge("neo4j_retrieval", END)

tourism_agent = workflow.compile()

print("Updated LangGraph workflow compiled successfully!")

### 8.13 Test the Updated LangGraph Workflow

We now test the workflow using both the natural-language question and a specific graph keyword.

ChromaDB will receive the complete question, while Neo4j will receive the extracted keyword.

In [ ]:
initial_state = {
    "question": "I want to see gorillas and enjoy nature",
    "graph_keyword": "gorilla",
    "vector_results": [],
    "graph_results": [],
    "final_answer": ""
}

result = tourism_agent.invoke(initial_state)

print("CHROMADB RESULTS")
print("=" * 50)

for item in result["vector_results"]:
    print(item["metadata"]["name"])

print("\nNEO4J RESULTS")
print("=" * 50)

for item in result["graph_results"]:
    print(item["destination"])

### 8.14 Design the Agent Routing Logic

An agent should be able to select the appropriate retrieval tool based on the user's question.

We introduce a routing decision that determines whether to use ChromaDB, Neo4j, or both.

This first implementation will use simple rule-based routing. Later, Qwen will replace these rules with intelligent tool selection.

### 8.15 Create the Agent Routing Function

The routing function determines which retrieval tool should be used based on the user's question.

It supports three possible decisions:

- `vector`: Use ChromaDB for semantic retrieval.
- `graph`: Use Neo4j for relationship-based retrieval.
- `both`: Use both retrieval systems.

This is our initial rule-based implementation of agent tool selection.

In [ ]:
def route_retrieval(state: TourismAgentState):

    question = state["question"].lower()

    graph_keywords = [
        "district", "region", "located",
        "connected", "relationships",
        "province", "where is"
    ]

    vector_keywords = [
        "recommend", "suggest", "similar",
        "describe", "experience", "explore"
    ]

    has_graph_keyword = any(
        keyword in question for keyword in graph_keywords
    )

    has_vector_keyword = any(
        keyword in question for keyword in vector_keywords
    )

    if has_graph_keyword and has_vector_keyword:
        return "both"

    elif has_graph_keyword:
        return "graph"

    elif has_vector_keyword:
        return "vector"

    else:
        return "both"

### 8.16 Test the Agent Routing Function

We test the routing function using three different tourism questions.

The purpose is to verify whether the function selects ChromaDB, Neo4j, or both retrieval tools appropriately.


In [ ]:
test_questions = [
    "Recommend destinations for nature lovers",
    "Which district is Volcanoes National Park located in?",
    "Recommend destinations near lakes and explain their activities"
]

for question in test_questions:

    state = {
        "question": question,
        "graph_keyword": "",
        "vector_results": [],
        "graph_results": [],
        "final_answer": ""
    }

    decision = route_retrieval(state)

    print("Question:", question)
    print("Selected tool:", decision)
    print("-" * 60)

### 8.17 Improve the Agent Routing Logic

We improve the rule-based routing function by adding tourism-related keywords.

This helps the agent recognize questions that require both semantic understanding and structured graph relationships.

In [ ]:
def route_retrieval(state: TourismAgentState):

    question = state["question"].lower()

    graph_keywords = [
        "district", "region", "located",
        "connected", "relationships",
        "province", "where is",
        "activities", "activity",
        "lake", "near", "region"
    ]

    vector_keywords = [
        "recommend", "suggest", "similar",
        "describe", "experience", "explore"
    ]

    has_graph_keyword = any(
        keyword in question for keyword in graph_keywords
    )

    has_vector_keyword = any(
        keyword in question for keyword in vector_keywords
    )

    if has_graph_keyword and has_vector_keyword:
        return "both"

    elif has_graph_keyword:
        return "graph"

    elif has_vector_keyword:
        return "vector"

    else:
        return "both"

In [ ]:
test_questions = [
    "Recommend destinations for nature lovers",
    "Which district is Volcanoes National Park located in?",
    "Recommend destinations near lakes and explain their activities"
]

for question in test_questions:

    state = {
        "question": question,
        "graph_keyword": "",
        "vector_results": [],
        "graph_results": [],
        "final_answer": ""
    }

    decision = route_retrieval(state)

    print("Question:", question)
    print("Selected tool:", decision)
    print("-" * 60)

### 8.18 Connect Conditional Routing to LangGraph

In this step, we connect our routing function to the LangGraph workflow using conditional edges.

The router determines which retrieval node or nodes should execute.

- vector: Execute ChromaDB.
- graph: Execute Neo4j.
- both: Execute both retrieval nodes.

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(TourismAgentState)

# Register nodes
workflow.add_node("chroma_retrieval", chroma_retrieval_node)
workflow.add_node("neo4j_retrieval", neo4j_retrieval_node)

# Connect the router to the retrieval nodes
workflow.add_conditional_edges(
    START,
    route_retrieval,
    {
        "vector": "chroma_retrieval",
        "graph": "neo4j_retrieval",
        "both": "chroma_retrieval"
    }
)

# Complete the individual retrieval paths
workflow.add_edge("chroma_retrieval", END)
workflow.add_edge("neo4j_retrieval", END)

tourism_agent = workflow.compile()

print("Conditional LangGraph workflow compiled successfully!")

### 8.19 Implement Parallel Retrieval

In this step, we configure LangGraph to execute ChromaDB and Neo4j in parallel when the router selects both.

The workflow will support three paths:

- vector: ChromaDB only.
- graph: Neo4j only.
- both: ChromaDB and Neo4j simultaneously.

After retrieval, both results will be available in the shared agent state.

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(TourismAgentState)

# Register retrieval nodes
workflow.add_node("chroma_retrieval", chroma_retrieval_node)
workflow.add_node("neo4j_retrieval", neo4j_retrieval_node)

# Route the user's question
workflow.add_conditional_edges(
    START,
    route_retrieval,
    {
        "vector": "chroma_retrieval",
        "graph": "neo4j_retrieval",
        "both": "chroma_retrieval"
    }
)

# Complete individual retrieval paths
workflow.add_edge("chroma_retrieval", END)
workflow.add_edge("neo4j_retrieval", END)

# Compile workflow
tourism_agent = workflow.compile()

print("Workflow compiled successfully!")

### 8.20 Implement Parallel Retrieval Using Send

LangGraph provides the Send mechanism to dynamically dispatch tasks to different nodes.

We will use it to execute ChromaDB and Neo4j simultaneously when both retrieval systems are required.

The results will be combined in the shared agent state.

In [ ]:
from typing import Annotated
from operator import add
from langgraph.types import Send

class TourismAgentState(TypedDict):
    question: str
    graph_keyword: str
    vector_results: Annotated[List[Dict[str, Any]], add]
    graph_results: Annotated[List[Dict[str, Any]], add]
    final_answer: str

### 8.21 Create the Parallel Retrieval Dispatcher

The dispatcher checks the routing decision and sends the question to the appropriate retrieval node.

- Vector: runs ChromaDB retrieval.
- Graph: runs Neo4j retrieval.
- Both: runs ChromaDB and Neo4j retrieval in parallel.


In [ ]:
from langgraph.types import Send

def dispatch_retrieval(state: TourismAgentState):

    decision = route_retrieval(state)

    if decision == "vector":
        return [Send("chroma_retrieval", state)]

    elif decision == "graph":
        return [Send("neo4j_retrieval", state)]

    else:
        return [
            Send("chroma_retrieval", state),
            Send("neo4j_retrieval", state)
        ]

### 8.22 Build the Parallel Retrieval Workflow

We will rebuild the LangGraph workflow using the parallel dispatcher.

The dispatcher will decide whether to execute ChromaDB, Neo4j, or both retrieval nodes. When both are selected, LangGraph will execute them in parallel and collect their results in the shared state.

In [ ]:
workflow = StateGraph(TourismAgentState)

workflow.add_node("chroma_retrieval", chroma_retrieval_node)
workflow.add_node("neo4j_retrieval", neo4j_retrieval_node)

workflow.add_conditional_edges(
    START,
    dispatch_retrieval
)

workflow.add_edge("chroma_retrieval", END)
workflow.add_edge("neo4j_retrieval", END)

tourism_agent = workflow.compile()

print("Parallel retrieval workflow compiled successfully!")

### 8.23 Test Parallel Retrieval

We will test a question that requires both semantic and graph-based retrieval.

The workflow should:
1. Search ChromaDB for semantically relevant destinations.
2. Search Neo4j for graph-related matches.
3. Collect both results in the shared state.

In [ ]:
test_state = {
    "question": "I want to visit lakes and enjoy water activities",
    "graph_keyword": "lake",
    "vector_results": [],
    "graph_results": [],
    "final_answer": ""
}

result = tourism_agent.invoke(test_state)

print("VECTOR RESULTS:")
for item in result["vector_results"]:
    print("-", item["metadata"]["name"])

print("\nGRAPH RESULTS:")
for item in result["graph_results"]:
    print("-", item["destination"])

### 8.24 Debug the ChromaDB Retrieval Node

We will execute the ChromaDB retrieval node independently to determine whether it returns results correctly.

In [ ]:
vector_test = chroma_retrieval_node({
    "question": "I want to visit lakes and enjoy water activities"
})

print("Number of vector results:", len(vector_test["vector_results"]))
print(vector_test["vector_results"])

### 8.25 Inspect the Parallel Workflow Output

We will inspect the complete state returned by LangGraph to identify why the ChromaDB results are missing after workflow execution.

In [ ]:
print("Returned state keys:", result.keys())

print("\nVector results:")
print(result.get("vector_results"))

print("\nGraph results:")
print(result.get("graph_results"))

### 8.26 Inspect the ChromaDB Node and Shared State

We will inspect the current ChromaDB node implementation and the shared state annotations to identify why its results are not being collected by LangGraph.

In [ ]:
import inspect
from typing import get_type_hints

print("CHROMA NODE:")
print(inspect.getsource(chroma_retrieval_node))

print("\nSHARED STATE:")
print(get_type_hints(TourismAgentState, include_extras=True))

### 8.27 Verify Parallel Task Dispatch

We will inspect the dispatcher's output to confirm that it creates two tasks: one for ChromaDB and one for Neo4j.

In [ ]:
dispatch_test = dispatch_retrieval(test_state)

print("Number of dispatched tasks:", len(dispatch_test))

for task in dispatch_test:
    print("Target node:", task.node)

### 8.28 Improve the Retrieval Routing Rules

We will update the routing rules to recognize singular and plural keywords, ensuring that questions requiring both semantic and graph retrieval are dispatched to both systems.

In [ ]:
def route_retrieval(state: TourismAgentState):

    question = state["question"].lower()

    graph_keywords = [
        "district", "region", "located",
        "connected", "relationships",
        "province", "where is",
        "activities", "activity",
        "lake", "lakes", "near"
    ]

    vector_keywords = [
        "recommend", "suggest", "similar",
        "describe", "experience", "explore"
    ]

    has_graph_keyword = any(
        keyword in question for keyword in graph_keywords
    )

    has_vector_keyword = any(
        keyword in question for keyword in vector_keywords
    )

    if has_graph_keyword and has_vector_keyword:
        return "both"
    elif has_graph_keyword:
        return "graph"
    elif has_vector_keyword:
        return "vector"
    else:
        return "both"

### 8.29 Verify the Updated Routing Decision

We will check whether the question about lakes and water activities is correctly classified as requiring both ChromaDB and Neo4j.

In [ ]:
print("Routing decision:", route_retrieval(test_state))

dispatch_test = dispatch_retrieval(test_state)

print("Number of dispatched tasks:", len(dispatch_test))

for task in dispatch_test:
    print("Target node:", task.node)

### 8.30 Improve Semantic and Graph Intent Detection

We will expand the vector keywords to recognize experience-oriented questions.

Questions containing both graph-related and semantic search intentions should route to both retrieval systems.

In [ ]:
def route_retrieval(state: TourismAgentState):

    question = state["question"].lower()

    graph_keywords = [
        "district", "region", "located",
        "connected", "relationships",
        "province", "where is",
        "activities", "activity",
        "lake", "lakes", "near"
    ]

    vector_keywords = [
        "recommend", "suggest", "similar",
        "describe", "experience", "explore",
        "visit", "enjoy", "water",
        "nature", "want"
    ]

    has_graph_keyword = any(
        keyword in question for keyword in graph_keywords
    )

    has_vector_keyword = any(
        keyword in question for keyword in vector_keywords
    )

    if has_graph_keyword and has_vector_keyword:
        return "both"
    elif has_graph_keyword:
        return "graph"
    elif has_vector_keyword:
        return "vector"
    else:
        return "both"

### 8.31 Verify the Updated Routing Decision

We will check whether the question is dispatched to both ChromaDB and Neo4j.

In [ ]:
print("Routing decision:", route_retrieval(test_state))

dispatch_test = dispatch_retrieval(test_state)

print("Number of dispatched tasks:", len(dispatch_test))

for task in dispatch_test:
    print("Target node:", task.node)

### 8.32 Rebuild the Parallel Retrieval Workflow

We will rebuild the workflow so it uses the updated routing rules.

When a question requires both retrieval systems, LangGraph will execute both nodes in parallel.

In [ ]:
workflow = StateGraph(TourismAgentState)

workflow.add_node("chroma_retrieval", chroma_retrieval_node)
workflow.add_node("neo4j_retrieval", neo4j_retrieval_node)

workflow.add_conditional_edges(
    START,
    dispatch_retrieval
)

workflow.add_edge("chroma_retrieval", END)
workflow.add_edge("neo4j_retrieval", END)

tourism_agent = workflow.compile()

print("Updated parallel workflow compiled successfully!")

### 8.33 Test the Updated Parallel Retrieval Workflow

We will execute the workflow using the same question about lakes and water activities.

We expect both ChromaDB and Neo4j to return results, which LangGraph will collect in the shared state.

In [ ]:
result = tourism_agent.invoke(test_state)

print("VECTOR RESULTS:")
for item in result["vector_results"]:
    print("-", item["metadata"]["name"])

print("\nGRAPH RESULTS:")
for item in result["graph_results"]:
    print("-", item["destination"])

print("\nTOTAL VECTOR RESULTS:", len(result["vector_results"]))
print("TOTAL GRAPH RESULTS:", len(result["graph_results"]))

### 8.34 Summary of Parallel Retrieval

We have successfully implemented parallel retrieval using LangGraph.

The workflow can now:

1. Route questions to ChromaDB for semantic retrieval.
2. Route questions to Neo4j for graph retrieval.
3. Execute both retrieval systems in parallel when required.
4. Collect results from both systems in the shared state.



## 9.1 Check GPU Availability

Before loading the Qwen language model, we will inspect the available hardware.

We will check:
- Whether a GPU is available.
- The GPU model.
- Available GPU memory.
- The installed PyTorch version.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

    print(f"GPU memory: {gpu_memory:.2f} GB")
else:
    print("No GPU detected. Consider enabling a GPU runtime.")

## 9.2 Install Hugging Face Transformers

We will use Hugging Face Transformers to load and run the Qwen language model in Google Colab.

We will also install Accelerate to help manage model loading and GPU placement.

In [ ]:
%pip install -q -U transformers accelerate

In [ ]:
import transformers
import accelerate

print("Transformers version:", transformers.__version__)
print("Accelerate version:", accelerate.__version__)

## 9.4 Select Qwen3 Model

We will use Qwen3-4B-Instruct-2507 as the language model for our Rwanda Tourism Assistant.

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

print("Selected model:", MODEL_NAME)

## 9.5 Load the Qwen3 Tokenizer

The tokenizer converts text into tokens that the Qwen3 model can understand.

We will load the tokenizer from Hugging Face using our selected model name.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Qwen3 tokenizer loaded successfully!")

## 9.6 Load the Qwen3 Model

We will load Qwen3-4B-Instruct-2507 using 4-bit quantization.

Quantization reduces GPU memory consumption while allowing us to run the model on our available Tesla T4 GPU.

In [ ]:
%pip install -q -U bitsandbytes

## 9.7 Verify Bitsandbytes Installation

Check the installed bitsandbytes version before configuring 4-bit quantization for Qwen3.

In [ ]:
import bitsandbytes as bnb

print("Bitsandbytes version:", bnb.__version__)
print("Bitsandbytes installed successfully!")

## 9.8 Configure 4-Bit Quantization

We will configure BitsAndBytes to load Qwen3-4B using 4-bit quantization.

We will use:
- NF4 quantization.
- Double quantization to reduce memory usage.
- Float16 computation for compatibility with the Tesla T4 GPU.

In [ ]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("4-bit quantization configured successfully!")

## 9.9 Load the Qwen3 Model

Load Qwen3-4B-Instruct-2507 using the tokenizer and 4-bit quantization configuration prepared earlier.

The model will be automatically placed on the available GPU.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Qwen3 model loaded successfully!")

## 9.10 Verify Qwen3 GPU Memory Usage

Check the GPU memory currently allocated and reserved by PyTorch after loading Qwen3.

This helps us confirm that the quantized model is running within the available Tesla T4 memory.

In [ ]:
allocated_memory = torch.cuda.memory_allocated(0) / (1024 ** 3)
reserved_memory = torch.cuda.memory_reserved(0) / (1024 ** 3)

print(f"GPU allocated memory: {allocated_memory:.2f} GB")
print(f"GPU reserved memory: {reserved_memory:.2f} GB")

## 9.11 Test Qwen3 Text Generation

We will send a sample question to Qwen3 and generate a response.

This test confirms that the model can process an instruction and generate a natural-language answer before we integrate it with ChromaDB, Neo4j, and LangGraph.

In [ ]:
messages = [
    {
        "role": "user",
        "content": "What are three popular tourist attractions in Rwanda?"
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.8
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("Qwen3 Response:\n")
print(response)

## 9.12 Prepare the RAG Prompt

We will create a prompt template that instructs Qwen3 to answer questions using retrieved context.

The prompt will:
- Use only the provided context.
- Avoid inventing unsupported information.
- Clearly indicate when the context does not contain an answer.
- Provide helpful and relevant tourism information.

In [ ]:
def create_rag_prompt(question, context):

    prompt = f"""
You are a Rwanda Tourism Assistant.

Answer the user's question using ONLY the information provided in the context.

Instructions:
1. Do not invent facts or destinations.
2. Do not use outside knowledge.
3. If the context does not contain enough information, say so.
4. Give clear, helpful and relevant answers.
5. Organize recommendations in a readable format.

Context:
{context}

User Question:
{question}

Answer:
"""

    return prompt


print("RAG prompt template created successfully!")

## 9.12 Prepare the Rwanda-Only RAG Prompt

The Rwanda Tourism Assistant must operate within Rwanda's geographical boundaries.

Instructions:
- Only provide tourism information about destinations located in Rwanda.
- Do not recommend destinations in Tanzania, Burundi, Kenya, Uganda, or other countries.
- If a user asks about another country, explain that the system currently covers Rwanda only.
- After explaining the limitation, suggest relevant destinations in Rwanda based on the user's original interests.
- Use only the retrieved context as evidence for destination recommendations.
- Never invent destination details.
- If no relevant Rwanda destination is found in the context, clearly say that the available dataset does not contain a suitable recommendation.

In [ ]:
def create_rag_prompt(question, context):

    prompt = f"""
You are a Rwanda Tourism Assistant.

GEOGRAPHICAL SCOPE:
You provide tourism information ONLY about Rwanda.

STRICT INSTRUCTIONS:

1. Only recommend tourist destinations located in Rwanda.
2. Never provide tourism recommendations for other countries.
3. If the user asks about Tanzania, Burundi, Kenya, Uganda,
   or any other country outside Rwanda:
   - Politely explain that this assistant currently covers Rwanda only.
   - Do not provide information about the requested foreign destination.
   - Suggest relevant destinations in Rwanda based on the user's
     original interests and activities.
4. Use ONLY the provided context when recommending destinations.
5. Never invent destinations, activities, prices, or other facts.
6. If the context does not contain relevant Rwanda information,
   explain that the available dataset does not provide a suitable answer.
7. Keep responses clear, helpful, and friendly.

RETRIEVED CONTEXT:
{context}

USER QUESTION:
{question}

ANSWER:
"""

    return prompt


print("Rwanda-only RAG prompt created successfully!")

## 9.13 Test Rwanda-Only Geographical Restrictions

Test the RAG prompt using a question about a destination outside Rwanda.

We expect Qwen3 to:
- Explain that the assistant only covers Rwanda.
- Avoid recommending foreign destinations.
- Suggest relevant Rwanda destinations using the supplied context.

In [ ]:
test_question = "What are the best tourist attractions in Tanzania?"

test_context = """
Volcanoes National Park:
Mountain gorilla trekking, hiking and nature experiences.

Akagera National Park:
Wildlife safaris and birdwatching.

Nyungwe Forest National Park:
Chimpanzee tracking, canopy walks and birdwatching.
"""

test_prompt = create_rag_prompt(
    question=test_question,
    context=test_context
)

messages = [
    {
        "role": "user",
        "content": test_prompt
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=250,
    do_sample=False
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

## 9.14 Retrieve and Combine Context from ChromaDB and Neo4j

We will use our existing LangGraph workflow to retrieve relevant tourism information.

The workflow will:
1. Receive a user's tourism question.
2. Route the question to ChromaDB, Neo4j, or both.
3. Retrieve relevant destinations.
4. Combine the retrieved information into a single context.

This context will be passed to Qwen3 for answer generation in the next step.

In [ ]:
# Test question
question = "I want to see gorillas and enjoy nature"

# Graph keyword for this test
graph_keyword = "gorilla"

# Run our existing LangGraph retrieval workflow
retrieval_result = tourism_agent.invoke({
    "question": question,
    "graph_keyword": graph_keyword,
    "vector_results": [],
    "graph_results": [],
    "final_answer": ""
})

# Collect retrieved results
vector_results = retrieval_result.get("vector_results", [])
graph_results = retrieval_result.get("graph_results", [])

# Combine and deduplicate destinations
combined_destinations = {}

for item in vector_results:
    destination_id = item["id"]

    combined_destinations[destination_id] = {
        "id": destination_id,
        "vector_data": item,
        "graph_data": None
    }

for item in graph_results:
    destination_id = item["id"]

    if destination_id in combined_destinations:
        combined_destinations[destination_id]["graph_data"] = item
    else:
        combined_destinations[destination_id] = {
            "id": destination_id,
            "vector_data": None,
            "graph_data": item
        }

# Build context from retrieved evidence
context_parts = []

for destination in combined_destinations.values():

    if destination["vector_data"]:
        vector_item = destination["vector_data"]

        context_parts.append(
            f"Destination ID: {destination['id']}\n"
            f"Vector document: {vector_item['document']}\n"
            f"Metadata: {vector_item['metadata']}"
        )

    if destination["graph_data"]:
        graph_item = destination["graph_data"]

        context_parts.append(
            f"Graph information:\n"
            f"Destination: {graph_item['destination']}\n"
            f"Type: {graph_item['type']}\n"
            f"Regions: {graph_item['regions']}\n"
            f"Districts: {graph_item['districts']}\n"
            f"Activities: {graph_item['activities']}\n"
            f"Interests: {graph_item['interests']}\n"
            f"Moods: {graph_item['moods']}"
        )

retrieved_context = "\n\n".join(context_parts)

print("Vector results:", len(vector_results))
print("Graph results:", len(graph_results))
print("Unique destinations:", len(combined_destinations))

print("\nRETRIEVED CONTEXT:\n")
print(retrieved_context)

## 9.15 Generate an Answer Using Retrieved Context

Pass the retrieved tourism information from ChromaDB and Neo4j to Qwen3.

The model must:
- Answer using the retrieved context only.
- Recommend only Rwanda destinations.
- Avoid inventing information.
- Explain when the retrieved information is insufficient.


In [ ]:
# Create the RAG prompt using actual retrieved information
rag_prompt = create_rag_prompt(
    question=question,
    context=retrieved_context
)

# Prepare the conversation
messages = [
    {
        "role": "user",
        "content": rag_prompt
    }
]

# Apply Qwen3 chat template
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

# Tokenize and move inputs to GPU
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

# Generate answer
outputs = model.generate(
    **inputs,
    max_new_tokens=400,
    do_sample=False
)

# Extract generated response
response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("QUESTION:")
print(question)

print("\nQWEN3 RAG ANSWER:")
print(response)

## 9.16 Create the Qwen3 Answer-Generation Node

Create a LangGraph node that receives the user's question and retrieved information from ChromaDB and Neo4j.

The node will:
- Combine retrieved destination information.
- Build the Rwanda-only RAG prompt.
- Pass the prompt to Qwen3.
- Generate the final tourism response.

The generated answer will be stored in the shared LangGraph state.

In [ ]:
def qwen_generation_node(state: TourismAgentState):

    # Collect retrieved information
    vector_results = state.get("vector_results", [])
    graph_results = state.get("graph_results", [])

    # Combine and deduplicate destinations
    combined_destinations = {}

    for item in vector_results:
        destination_id = item["id"]

        combined_destinations[destination_id] = {
            "vector_data": item,
            "graph_data": None
        }

    for item in graph_results:
        destination_id = item["id"]

        if destination_id in combined_destinations:
            combined_destinations[destination_id]["graph_data"] = item
        else:
            combined_destinations[destination_id] = {
                "vector_data": None,
                "graph_data": item
            }

    # Build retrieved context
    context_parts = []

    for destination_id, destination in combined_destinations.items():

        if destination["vector_data"]:
            item = destination["vector_data"]

            context_parts.append(
                f"Destination ID: {destination_id}\n"
                f"Document: {item['document']}\n"
                f"Metadata: {item['metadata']}"
            )

        if destination["graph_data"]:
            item = destination["graph_data"]

            context_parts.append(
                f"Graph information:\n"
                f"Destination: {item['destination']}\n"
                f"Type: {item['type']}\n"
                f"Regions: {item['regions']}\n"
                f"Districts: {item['districts']}\n"
                f"Activities: {item['activities']}\n"
                f"Interests: {item['interests']}\n"
                f"Moods: {item['moods']}"
            )

    retrieved_context = "\n\n".join(context_parts)

    # Create Rwanda-only RAG prompt
    rag_prompt = create_rag_prompt(
        question=state["question"],
        context=retrieved_context
    )

    # Prepare Qwen3 conversation
    messages = [
        {
            "role": "user",
            "content": rag_prompt
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    # Generate final answer
    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return {
        "final_answer": answer
    }


print("Qwen3 generation node created successfully!")

## 9.17 Integrate Qwen3 into LangGraph

Update the LangGraph workflow to include the Qwen3 answer-generation node.

The workflow will:
1. Route the user's question to ChromaDB, Neo4j, or both.
2. Execute the selected retrieval operations.
3. Pass the retrieved results to Qwen3.
4. Return the final generated answer.

When both retrieval systems are selected, they execute in parallel before Qwen3 runs.

In [ ]:
# Create the updated LangGraph workflow
workflow = StateGraph(TourismAgentState)

# Register retrieval and generation nodes
workflow.add_node("chroma_retrieval", chroma_retrieval_node)
workflow.add_node("neo4j_retrieval", neo4j_retrieval_node)
workflow.add_node("qwen_generation", qwen_generation_node)

# Route to the selected retrieval nodes
workflow.add_conditional_edges(
    START,
    dispatch_retrieval
)

# Both retrieval branches lead to Qwen3
workflow.add_edge("chroma_retrieval", "qwen_generation")
workflow.add_edge("neo4j_retrieval", "qwen_generation")

# Finish after answer generation
workflow.add_edge("qwen_generation", END)

# Compile the workflow
tourism_agent = workflow.compile()

print("Complete Agentic RAG workflow compiled successfully!")

## 9.18 Test the Complete Agentic RAG Workflow

Test the integrated Rwanda Tourism Assistant.

The test will:
1. Receive a natural-language tourism question.
2. Select the appropriate retrieval tools.
3. Retrieve information from ChromaDB and Neo4j.
4. Combine the retrieved evidence.
5. Generate the final answer using Qwen3.

We will inspect the retrieval results and the generated answer.

In [ ]:
test_input = {
    "question": "where can i go if i want to do carting",
    "graph_keyword": "gorilla",
    "vector_results": [],
    "graph_results": [],
    "final_answer": ""
}

result = tourism_agent.invoke(test_input)

print("QUESTION:")
print(result["question"])

print("\nVECTOR RESULTS:", len(result["vector_results"]))
print("GRAPH RESULTS:", len(result["graph_results"]))

print("\nFINAL QWEN3 ANSWER:\n")
print(result["final_answer"])

## 10.1 Define Geographical Scope Validation

The Rwanda Tourism Assistant operates exclusively within Rwanda.

Before retrieving information, the system must identify whether the user's question targets another country.

Rules:

1. Rwanda is the only supported geographical scope.
2. Questions about Rwanda proceed to normal retrieval.
3. Questions explicitly targeting other countries are marked as outside scope.
4. The original tourism interests must be preserved.
5. For outside-country questions, retrieve relevant alternatives from our Rwanda dataset.
6. If no suitable Rwanda alternative is found, explain that the available dataset does not contain a relevant recommendation.

The validation stage will run before ChromaDB and Neo4j retrieval.

## 10.2 Define Supported Geographical Scope

Create a list of country names that the assistant should recognize.

Rwanda is the only supported country. Other recognized countries will trigger the outside-scope handling process.

In [ ]:
SUPPORTED_COUNTRY = "rwanda"

KNOWN_COUNTRIES = {
    "rwanda",
    "tanzania",
    "burundi",
    "uganda",
    "kenya",
    "democratic republic of the congo",
    "dr congo",
    "south sudan",
    "ethiopia",
    "somalia",
    "malawi",
    "zambia",
    "zimbabwe",
    "south africa",
    "nigeria",
    "ghana",
    "egypt",
    "morocco",
    "united states",
    "united kingdom"
}

print("Supported country:", SUPPORTED_COUNTRY.title())
print("Recognized countries:", len(KNOWN_COUNTRIES))

## 10.3 Create the Geographical Validation Function

Create a function that checks whether a user's tourism question explicitly mentions a country outside Rwanda.

The function returns:
- `in_scope`: Whether the question is within Rwanda's geographical scope.
- `detected_country`: The detected country, if any.
- `original_question`: The user's original question.

This is an initial rule-based implementation using country-name matching.

In [ ]:
def validate_geographical_scope(question):

    normalized_question = question.lower()

    detected_country = None

    # Check for recognized country names
    for country in KNOWN_COUNTRIES:
        if country in normalized_question:
            detected_country = country
            break

    # Rwanda is supported
    if detected_country == SUPPORTED_COUNTRY:
        return {
            "in_scope": True,
            "detected_country": detected_country,
            "original_question": question
        }

    # Another recognized country is outside scope
    elif detected_country is not None:
        return {
            "in_scope": False,
            "detected_country": detected_country,
            "original_question": question
        }

    # No country explicitly mentioned
    return {
        "in_scope": True,
        "detected_country": None,
        "original_question": question
    }


print("Geographical validation function created successfully!")

## 10.4 Test Geographical Validation

Test the validation function using different geographical questions.

Expected behaviour:
- Rwanda questions: In scope.
- Tanzania questions: Outside scope.
- Burundi questions: Outside scope.
- Questions without a country: In scope by default.


In [ ]:
test_questions = [
    "Recommend tourist attractions in Rwanda",
    "What are the best places to visit in Tanzania?",
    "What can I do in Burundi?",
    "I want to see gorillas and enjoy nature"
]

for question in test_questions:

    result = validate_geographical_scope(question)

    print("Question:", question)
    print("In scope:", result["in_scope"])
    print("Detected country:", result["detected_country"])
    print("-" * 60)

## 10.5 Update the LangGraph Shared State

Extend TourismAgentState with geographical validation fields.

New fields:
- `in_scope`: Indicates whether the request is within Rwanda's geographical scope.
- `detected_country`: Stores the country identified in the question.

These fields will allow LangGraph to make routing decisions before executing retrieval tools.

In [ ]:
class TourismAgentState(TypedDict):
    question: str
    graph_keyword: str
    in_scope: bool
    detected_country: str
    vector_results: Annotated[List[Dict[str, Any]], add]
    graph_results: Annotated[List[Dict[str, Any]], add]
    final_answer: str

print("TourismAgentState updated successfully!")

## 10.6 Create the Geographical Validation Node

Create a LangGraph node that calls the geographical validation function.

The node will:
1. Receive the user's question.
2. Check whether the question targets a country outside Rwanda.
3. Store the validation result in the shared state.
4. Pass the updated state to the next workflow stage.

In [ ]:
def geographical_validation_node(state: TourismAgentState):

    result = validate_geographical_scope(state["question"])

    print("Geographical validation:")
    print("In scope:", result["in_scope"])
    print("Detected country:", result["detected_country"])

    return {
        "in_scope": result["in_scope"],
        "detected_country": result["detected_country"]
    }


print("Geographical validation node created successfully!")

### 10.7 Updating the Agent State

We introduce `retrieval_question` to separate the user's original question from the query used to retrieve Rwanda tourism information.

This allows the agent to preserve the original question while searching only for relevant destinations in Rwanda.


In [ ]:
class TourismAgentState(TypedDict):
    question: str
    retrieval_question: str
    graph_keyword: str
    in_scope: bool
    detected_country: str
    vector_results: Annotated[List[Dict[str, Any]], add]
    graph_results: Annotated[List[Dict[str, Any]], add]
    final_answer: str

print("TourismAgentState updated successfully!")

This function prepares a retrieval query for our Rwanda-only tourism database.

For out-of-scope questions, it removes the detected foreign country name so that retrieval focuses on the user's tourism interests rather than destinations outside Rwanda.

In [ ]:
import re

def prepare_retrieval_question(question, detected_country):

    if detected_country is None or detected_country == SUPPORTED_COUNTRY:
        return question

    pattern = re.compile(
        rf"\b{re.escape(detected_country)}\b",
        re.IGNORECASE
    )

    retrieval_question = pattern.sub("", question)

    return " ".join(retrieval_question.split())


print("Retrieval query preparation function created successfully!")

We test the function using three questions:
1. A question about Tanzania.
2. A question about Burundi.
3. A question without a foreign country.

The expected result is that foreign country names are removed while the original tourism interests remain.

In [ ]:
test_questions = [
    "What are the best places for gorilla trekking in Tanzania?",
    "What can I do for cultural tourism in Burundi?",
    "I want to see gorillas and enjoy nature"
]

for question in test_questions:

    validation = validate_geographical_scope(question)

    retrieval_question = prepare_retrieval_question(
        question,
        validation["detected_country"]
    )

    print("Original:", question)
    print("Detected country:", validation["detected_country"])
    print("Retrieval question:", retrieval_question)
    print("-" * 60)

Instead of simply removing the foreign country name, we replace it with Rwanda.

This produces a complete, Rwanda-focused retrieval question while preserving the user's original tourism interests.


In [ ]:
def prepare_retrieval_question(question, detected_country):

    if detected_country is None or detected_country == SUPPORTED_COUNTRY:
        return question

    pattern = re.compile(
        rf"\b{re.escape(detected_country)}\b",
        re.IGNORECASE
    )

    retrieval_question = pattern.sub("Rwanda", question)

    return " ".join(retrieval_question.split())


print("Retrieval query preparation function improved successfully!")

We test the improved function to confirm that:
- Tanzania is replaced with Rwanda.
- Burundi is replaced with Rwanda.
- Questions without foreign countries remain unchanged.

In [ ]:
for question in test_questions:

    validation = validate_geographical_scope(question)

    retrieval_question = prepare_retrieval_question(
        question,
        validation["detected_country"]
    )

    print("Original:", question)
    print("Detected country:", validation["detected_country"])
    print("Retrieval question:", retrieval_question)
    print("-" * 60)

Update the ChromaDB retrieval node to use the Rwanda-focused retrieval question.


In [ ]:
def chroma_retrieval_node(state: TourismAgentState):

    results = hybrid_retrieval(
        query=state["retrieval_question"],
        top_k=3
    )

    vector_results = []

    for i, document in enumerate(results["documents"][0]):
        vector_results.append({
            "id": results["ids"][0][i],
            "document": document,
            "metadata": results["metadatas"][0][i]
        })

    return {"vector_results": vector_results}


print("ChromaDB retrieval node updated successfully!")

Update the Neo4j retrieval node to use the Rwanda-focused retrieval question when searching the graph database.

In [ ]:
def neo4j_retrieval_node(state: TourismAgentState):

    results = neo4j_graph_retrieval(
        search_term=state["graph_keyword"],
        top_k=3
    )

    return {"graph_results": results}


print("Neo4j retrieval node updated successfully!")

Update the rule-based routing function to analyze the retrieval question rather than the original user question.

The agent will use this question to decide whether to search ChromaDB, Neo4j, or both.

In [ ]:
def route_retrieval(state: TourismAgentState):

    question = state["retrieval_question"].lower()

    graph_keywords = [
        "district", "region", "located",
        "connected", "relationships",
        "province", "where is",
        "activities", "activity", "lake", "near"
    ]

    vector_keywords = [
        "recommend", "suggest", "similar",
        "describe", "experience", "explore",
        "visit", "enjoy", "water", "nature",
        "want", "gorilla", "cultural"
    ]

    has_graph_keyword = any(
        keyword in question for keyword in graph_keywords
    )

    has_vector_keyword = any(
        keyword in question for keyword in vector_keywords
    )

    if has_graph_keyword and has_vector_keyword:
        return "both"
    elif has_graph_keyword:
        return "graph"
    elif has_vector_keyword:
        return "vector"
    else:
        return "both"


print("Retrieval routing function updated successfully!")

Test the routing function with three different tourism questions:

1. Nature and gorilla recommendations.
2. A geographical question about a district.
3. A question involving lakes and activities.

We expect the agent to select the appropriate retrieval source.

In [ ]:
routing_tests = [
    {
        "retrieval_question": "I want to see gorillas and enjoy nature",
        "expected": "vector"
    },
    {
        "retrieval_question": "Which district is Volcanoes National Park located in?",
        "expected": "graph"
    },
    {
        "retrieval_question": "Recommend lakes where I can enjoy water activities",
        "expected": "both"
    }
]

for test in routing_tests:

    decision = route_retrieval(test)

    print("Question:", test["retrieval_question"])
    print("Expected:", test["expected"])
    print("Actual:", decision)
    print("Correct:", decision == test["expected"])
    print("-" * 60)

The geographical routing dispatcher controls which retrieval nodes execute after geographical validation.

For supported Rwanda questions, it uses the normal retrieval decision.

For questions about countries outside Rwanda, it retrieves relevant alternatives from the Rwanda tourism database using both retrieval systems.

In [ ]:
from langgraph.types import Send

def geographical_dispatcher(state: TourismAgentState):

    if not state["in_scope"]:
        return [
            Send("chroma_retrieval", state),
            Send("neo4j_retrieval", state)
        ]

    return dispatch_retrieval(state)


print("Geographical routing dispatcher created successfully!")

Connect the geographical validation node, retrieval dispatcher, ChromaDB retrieval node, Neo4j retrieval node, and Qwen3 generation node.

Geographical validation must execute first. Retrieval then follows the geographical dispatcher, and Qwen3 generates the final answer after retrieval completes.

In [ ]:
workflow = StateGraph(TourismAgentState)

workflow.add_node(
    "geographical_validation",
    geographical_validation_node
)

workflow.add_node(
    "chroma_retrieval",
    chroma_retrieval_node
)

workflow.add_node(
    "neo4j_retrieval",
    neo4j_retrieval_node
)

workflow.add_node(
    "qwen_generation",
    qwen_generation_node
)

workflow.add_edge(
    START,
    "geographical_validation"
)

workflow.add_conditional_edges(
    "geographical_validation",
    geographical_dispatcher
)

workflow.add_edge("chroma_retrieval", "qwen_generation")
workflow.add_edge("neo4j_retrieval", "qwen_generation")
workflow.add_edge("qwen_generation", END)

tourism_agent = workflow.compile()

print("Geographical validation workflow compiled successfully!")

Test the complete LangGraph workflow with a question about gorilla trekking in Tanzania.

Expected behavior:
1. Geographical validation detects Tanzania.
2. The retrieval question is changed to focus on Rwanda.
3. Both ChromaDB and Neo4j retrieve Rwanda destinations.
4. Qwen3 explains that the assistant covers Rwanda only and suggests relevant Rwanda alternatives.

In [ ]:
test_state = {
    "question": "What are the best places for gorilla trekking in Tanzania?",
    "retrieval_question": "",
    "graph_keyword": "gorilla",
    "in_scope": True,
    "detected_country": None,
    "vector_results": [],
    "graph_results": [],
    "final_answer": ""
}

test_result = tourism_agent.invoke(test_state)

print("DETECTED COUNTRY:")
print(test_result["detected_country"])

print("\nRETRIEVAL QUESTION:")
print(test_result["retrieval_question"])

print("\nVECTOR RESULTS:")
for item in test_result["vector_results"]:
    print(item["metadata"]["name"])

print("\nGRAPH RESULTS:")
for item in test_result["graph_results"]:
    print(item["destination"])

print("\nFINAL ANSWER:")
print(test_result["final_answer"])

Rebuild the LangGraph workflow using the updated geographical validation node.

This ensures that the retrieval question is populated before ChromaDB and Neo4j execute.

In [ ]:
workflow = StateGraph(TourismAgentState)

workflow.add_node(
    "geographical_validation",
    geographical_validation_node
)

workflow.add_node("chroma_retrieval", chroma_retrieval_node)
workflow.add_node("neo4j_retrieval", neo4j_retrieval_node)
workflow.add_node("qwen_generation", qwen_generation_node)

workflow.add_edge(START, "geographical_validation")

workflow.add_conditional_edges(
    "geographical_validation",
    geographical_dispatcher
)

workflow.add_edge("chroma_retrieval", "qwen_generation")
workflow.add_edge("neo4j_retrieval", "qwen_generation")
workflow.add_edge("qwen_generation", END)

tourism_agent = workflow.compile()

print("Updated geographical workflow compiled successfully!")

In [ ]:
test_state = {
    "question": "What are the best places for gorilla trekking in Tanzania?",
    "retrieval_question": "",
    "graph_keyword": "gorilla",
    "in_scope": True,
    "detected_country": None,
    "vector_results": [],
    "graph_results": [],
    "final_answer": ""
}

test_result = tourism_agent.invoke(test_state)

print("DETECTED COUNTRY:")
print(test_result["detected_country"])

print("\nRETRIEVAL QUESTION:")
print(test_result["retrieval_question"])

print("\nVECTOR RESULTS:")
for item in test_result["vector_results"]:
    print(item["metadata"]["name"])

print("\nGRAPH RESULTS:")
for item in test_result["graph_results"]:
    print(item["destination"])

print("\nFINAL ANSWER:")
print(test_result["final_answer"])

Update the geographical validation node so it returns all three fields:
- in_scope
- detected_country
- retrieval_question

The retrieval question must be prepared before the workflow dispatches retrieval tasks.

In [ ]:
def geographical_validation_node(state: TourismAgentState):

    result = validate_geographical_scope(state["question"])

    retrieval_question = prepare_retrieval_question(
        state["question"],
        result["detected_country"]
    )

    print("Geographical validation:")
    print("In scope:", result["in_scope"])
    print("Detected country:", result["detected_country"])
    print("Retrieval question:", retrieval_question)

    return {
        "in_scope": result["in_scope"],
        "detected_country": result["detected_country"],
        "retrieval_question": retrieval_question
    }


print("Updated geographical validation node created!")

Recompile the tourism agent workflow with the corrected geographical validation node.

In [ ]:
workflow = StateGraph(TourismAgentState)

workflow.add_node("geographical_validation", geographical_validation_node)
workflow.add_node("chroma_retrieval", chroma_retrieval_node)
workflow.add_node("neo4j_retrieval", neo4j_retrieval_node)
workflow.add_node("qwen_generation", qwen_generation_node)

workflow.add_edge(START, "geographical_validation")

workflow.add_conditional_edges(
    "geographical_validation",
    geographical_dispatcher
)

workflow.add_edge("chroma_retrieval", "qwen_generation")
workflow.add_edge("neo4j_retrieval", "qwen_generation")
workflow.add_edge("qwen_generation", END)

tourism_agent = workflow.compile()

print("Workflow recompiled with updated geographical validation!")

Test the complete workflow using a question about gorilla trekking in Tanzania.

Verify that geographical validation detects Tanzania, rewrites the retrieval question to Rwanda, and retrieves relevant Rwanda destinations.

In [ ]:
test_state = {
    "question": "What are the best places for gorilla trekking in Tanzania?",
    "retrieval_question": "",
    "graph_keyword": "gorilla",
    "in_scope": True,
    "detected_country": None,
    "vector_results": [],
    "graph_results": [],
    "final_answer": ""
}

test_result = tourism_agent.invoke(test_state)

print("\nDETECTED COUNTRY:")
print(test_result["detected_country"])

print("\nRETRIEVAL QUESTION:")
print(test_result["retrieval_question"])

print("\nVECTOR RESULTS:")
for item in test_result["vector_results"]:
    print(item["metadata"]["name"])

print("\nGRAPH RESULTS:")
for item in test_result["graph_results"]:
    print(item["destination"])

print("\nFINAL ANSWER:")
print(test_result["final_answer"])

### 10.19 Testing Cultural Tourism for an Out-of-Scope Country

Test the agent using a question about cultural tourism in Burundi.

Expected behavior:
1. Detect Burundi as an unsupported country.
2. Rewrite the retrieval question to Rwanda.
3. Retrieve relevant Rwanda cultural destinations.
4. Generate an answer explaining the geographical limitation and suggesting Rwanda alternatives.

In [ ]:
test_state = {
    "question": "What can I do for cultural tourism in Burundi?",
    "retrieval_question": "",
    "graph_keyword": "culture",
    "in_scope": True,
    "detected_country": None,
    "vector_results": [],
    "graph_results": [],
    "final_answer": ""
}

test_result = tourism_agent.invoke(test_state)

print("DETECTED COUNTRY:")
print(test_result["detected_country"])

print("\nRETRIEVAL QUESTION:")
print(test_result["retrieval_question"])

print("\nVECTOR RESULTS:")
for item in test_result["vector_results"]:
    print(item["metadata"]["name"])

print("\nGRAPH RESULTS:")
for item in test_result["graph_results"]:
    print(item["destination"])

print("\nFINAL ANSWER:")
print(test_result["final_answer"])

### 10.20 Testing an In-Scope Rwanda Question

Test the agent using a question about gorilla trekking in Rwanda.

Expected behavior:
1. Detect Rwanda as the supported country.
2. Mark the question as in scope.
3. Keep the original retrieval question unchanged.
4. Execute normal retrieval routing.
5. Generate a Rwanda tourism recommendation.

In [ ]:
test_state = {
    "question": "Where can I go for gorilla trekking in Rwanda?",
    "retrieval_question": "",
    "graph_keyword": "gorilla",
    "in_scope": True,
    "detected_country": None,
    "vector_results": [],
    "graph_results": [],
    "final_answer": ""
}

test_result = tourism_agent.invoke(test_state)

print("DETECTED COUNTRY:")
print(test_result["detected_country"])

print("\nIN SCOPE:")
print(test_result["in_scope"])

print("\nRETRIEVAL QUESTION:")
print(test_result["retrieval_question"])

print("\nVECTOR RESULTS:")
for item in test_result["vector_results"]:
    print(item["metadata"]["name"])

print("\nGRAPH RESULTS:")
for item in test_result["graph_results"]:
    print(item["destination"])

print("\nFINAL ANSWER:")
print(test_result["final_answer"])

Use the pycountry library to recognize countries using their official,
common and standard names.

Add commonly used country aliases such as USA, UK, DRC and UAE.

Use case-insensitive whole-word matching to reduce false detections.
Rwanda remains the only supported tourism destination country.

In [ ]:
%pip install -q pycountry

import pycountry
import re

SUPPORTED_COUNTRY = "rwanda"

KNOWN_COUNTRIES = set()

for country in pycountry.countries:
    KNOWN_COUNTRIES.add(country.name.lower())

    if hasattr(country, "official_name"):
        KNOWN_COUNTRIES.add(country.official_name.lower())

    if hasattr(country, "common_name"):
        KNOWN_COUNTRIES.add(country.common_name.lower())


COUNTRY_ALIASES = {
    "usa": "united states",
    "us": "united states",
    "uk": "united kingdom",
    "britain": "united kingdom",
    "great britain": "united kingdom",
    "drc": "democratic republic of the congo",
    "dr congo": "democratic republic of the congo",
    "ivory coast": "cote d'ivoire",
    "south korea": "korea, republic of",
    "north korea": "korea, democratic people's republic of",
    "uae": "united arab emirates"
}

KNOWN_COUNTRIES.update(COUNTRY_ALIASES.keys())

def validate_geographical_scope(question):

    normalized_question = question.lower()
    detected_country = None

    for country in sorted(KNOWN_COUNTRIES, key=len, reverse=True):

        pattern = rf"(?<!\w){re.escape(country)}(?!\w)"

        if re.search(pattern, normalized_question):

            detected_country = COUNTRY_ALIASES.get(country, country)
            break

    if detected_country == SUPPORTED_COUNTRY:
        in_scope = True
    elif detected_country is not None:
        in_scope = False
    else:
        in_scope = True

    return {
        "in_scope": in_scope,
        "detected_country": detected_country,
        "original_question": question
    }


print("Recognized country names:", len(KNOWN_COUNTRIES))
print("Supported tourism country:", SUPPORTED_COUNTRY)
print("Country validation updated successfully!")

### 10.22 Testing Comprehensive Country Detection

Test country recognition using Rwanda and several unsupported countries,
including countries outside our original manually defined list.

Verify that Rwanda is accepted and all other recognized countries
are marked out of scope.

In [ ]:
test_countries = [
    "Recommend places to visit in Rwanda",
    "I want to visit Tanzania",
    "Show me attractions in Japan",
    "What can I do in Brazil?",
    "Recommend destinations in UAE",
    "What are the best places in DRC?"
]

for question in test_countries:

    result = validate_geographical_scope(question)

    print("Question:", question)
    print("Detected:", result["detected_country"])
    print("In scope:", result["in_scope"])
    print("-" * 50)

# PHASE 6 — EVALUATION

We evaluate the Rwanda Tourism Assistant across five dimensions:

1. Retrieval Quality
   - Are the retrieved destinations relevant to the question?

2. Answer Accuracy
   - Does Qwen3 generate answers supported by the dataset?

3. Hallucination Detection
   - Does the model invent destinations, activities, prices or facts?

4. Tool Selection
   - Does the agent correctly select ChromaDB, Neo4j or both?

5. Response Time
   - How long does the complete pipeline take to generate an answer?

The evaluation will use predefined test questions and measurable results.

Prepare test cases covering wildlife, culture, lakes, hiking,
geographical relationships and unsupported countries.

Each test case includes the expected retrieval strategy and
relevant destination where known.

In [ ]:
evaluation_cases = [
    {
        "question": "I want to see gorillas and enjoy nature",
        "graph_keyword": "gorilla",
        "expected_route": "vector",
        "expected_destination": "Volcanoes National Park"
    },
    {
        "question": "Which district is Volcanoes National Park located in?",
        "graph_keyword": "Volcanoes",
        "expected_route": "graph",
        "expected_destination": "Volcanoes National Park"
    },
    {
        "question": "Recommend lakes where I can enjoy water activities",
        "graph_keyword": "lake",
        "expected_route": "both",
        "expected_destination": "Lake Kivu - Karongi"
    },
    {
        "question": "Where can I experience cultural tourism in Rwanda?",
        "graph_keyword": "culture",
        "expected_route": "vector",
        "expected_destination": "Ethnographic Museum of Rwanda, Huye"
    },
    {
        "question": "What are the best places for gorilla trekking in Tanzania?",
        "graph_keyword": "gorilla",
        "expected_route": "both",
        "expected_destination": "Volcanoes National Park"
    }
]

print("Evaluation test cases:", len(evaluation_cases))

for index, case in enumerate(evaluation_cases, start=1):
    print(f"{index}. {case['question']}")

Evaluate the agent's routing decisions against the expected retrieval
strategy for each test case.

A routing decision is correct when the actual selected strategy
matches the expected strategy.

In [ ]:
routing_results = []

for case in evaluation_cases:

    actual_route = route_retrieval({
        "question": case["question"],
        "retrieval_question": case["question"]
    })

    expected_route = case["expected_route"]

    routing_results.append({
        "question": case["question"],
        "expected_route": expected_route,
        "actual_route": actual_route,
        "correct": expected_route == actual_route
    })

for result in routing_results:
    print("Question:", result["question"])
    print("Expected:", result["expected_route"])
    print("Actual:", result["actual_route"])
    print("Correct:", result["correct"])
    print("-" * 50)

accuracy = (
    sum(result["correct"] for result in routing_results)
    / len(routing_results)
) * 100

print(f"Tool Selection Accuracy: {accuracy:.2f}%")

### 6.3.1 — Evaluate Complete Routing Logic

Evaluate tool selection using the geographical validation
and retrieval routing decisions.

Foreign-country requests must trigger both retrieval tools
to identify relevant Rwanda alternatives.

In [ ]:
routing_results = []

for case in evaluation_cases:

    scope = validate_geographical_scope(case["question"])

    if not scope["in_scope"]:
        actual_route = "both"
    else:
        actual_route = route_retrieval({
            "question": case["question"],
            "retrieval_question": case["question"]
        })

    expected_route = case["expected_route"]

    routing_results.append({
        "question": case["question"],
        "expected_route": expected_route,
        "actual_route": actual_route,
        "correct": expected_route == actual_route
    })

for result in routing_results:
    print("Question:", result["question"])
    print("Expected:", result["expected_route"])
    print("Actual:", result["actual_route"])
    print("Correct:", result["correct"])
    print("-" * 50)

accuracy = (
    sum(result["correct"] for result in routing_results)
    / len(routing_results)
) * 100

print(f"Tool Selection Accuracy: {accuracy:.2f}%")

### 6.4 — Retrieval Quality Evaluation

Evaluate ChromaDB and Neo4j independently using the predefined
test cases.

Use Precision@3 and Recall@3 to measure retrieval quality
against the expected relevant destinations.

Only destinations supported by the Rwanda tourism dataset
will be considered relevant.

### 6.4.1 — Measure Retrieval Quality

For each evaluation question:

1. Validate its geographical scope.
2. Rewrite foreign-country questions for Rwanda.
3. Retrieve the top three results from ChromaDB.
4. Retrieve the top three results from Neo4j.
5. Compare retrieved destinations against the expected destination.

Calculate:

Precision@3 = Relevant retrieved results / 3

Recall@3 = Relevant retrieved results / Expected relevant results

For this initial evaluation, each test case has one expected
relevant destination.

In [ ]:
retrieval_results = []

for case in evaluation_cases:

    question = case["question"]
    expected = case["expected_destination"]

    # Apply geographical validation
    scope = validate_geographical_scope(question)

    retrieval_question = prepare_retrieval_question(
        question,
        scope["detected_country"]
    )

    # ChromaDB retrieval
    vector_data = hybrid_retrieval(
        query=retrieval_question,
        top_k=3
    )

    vector_names = [
        metadata.get("name", "")
        for metadata in vector_data["metadatas"][0]
    ]

    # Neo4j retrieval
    graph_data = neo4j_graph_retrieval(
        search_term=case["graph_keyword"],
        top_k=3
    )

    graph_names = [
        item["destination"]
        for item in graph_data
    ]

    # Calculate metrics
    for system, names in [
        ("ChromaDB", vector_names),
        ("Neo4j", graph_names)
    ]:

        relevant_count = sum(
            1 for name in names
            if name == expected
        )

        precision = relevant_count / 3
        recall = relevant_count / 1

        retrieval_results.append({
            "question": question,
            "system": system,
            "expected": expected,
            "retrieved": names,
            "precision_at_3": precision,
            "recall_at_3": recall
        })

        print("Question:", question)
        print("System:", system)
        print("Expected:", expected)
        print("Retrieved:", names)
        print(f"Precision@3: {precision:.2f}")
        print(f"Recall@3: {recall:.2f}")
        print("-" * 60)

### 6.4.2 — Normalize Destination Names

Some destination names include additional local names in parentheses.

Normalize names by removing parenthetical descriptions,
converting them to lowercase and trimming whitespace.

This allows equivalent destination names to match during evaluation.

In [ ]:
import re

def normalize_destination_name(name):

    name = re.sub(r"\s*\([^)]*\)", "", name)

    return name.strip().lower()


retrieval_results = []

for case in evaluation_cases:

    question = case["question"]
    expected = case["expected_destination"]

    scope = validate_geographical_scope(question)

    retrieval_question = prepare_retrieval_question(
        question,
        scope["detected_country"]
    )

    vector_data = hybrid_retrieval(
        query=retrieval_question,
        top_k=3
    )

    vector_names = [
        metadata.get("name", "")
        for metadata in vector_data["metadatas"][0]
    ]

    graph_data = neo4j_graph_retrieval(
        search_term=case["graph_keyword"],
        top_k=3
    )

    graph_names = [
        item["destination"]
        for item in graph_data
    ]

    expected_name = normalize_destination_name(expected)

    for system, names in [
        ("ChromaDB", vector_names),
        ("Neo4j", graph_names)
    ]:

        relevant_count = sum(
            normalize_destination_name(name) == expected_name
            for name in names
        )

        precision = relevant_count / 3
        recall = min(relevant_count / 1, 1)

        retrieval_results.append({
            "question": question,
            "system": system,
            "precision_at_3": precision,
            "recall_at_3": recall
        })

        print("Question:", question)
        print("System:", system)
        print("Expected:", expected)
        print("Retrieved:", names)
        print(f"Precision@3: {precision:.2f}")
        print(f"Recall@3: {recall:.2f}")
        print("-" * 60)

### 6.4.3 — Average Retrieval Performance

Calculate the average Precision@3 and Recall@3 for ChromaDB
and Neo4j across all evaluation questions.

Compare both retrieval systems to identify their strengths
and weaknesses.

In [ ]:
import pandas as pd

results_df = pd.DataFrame(retrieval_results)

summary = results_df.groupby("system").agg(
    average_precision=("precision_at_3", "mean"),
    average_recall=("recall_at_3", "mean")
).reset_index()

summary["average_precision"] *= 100
summary["average_recall"] *= 100

print("RETRIEVAL PERFORMANCE SUMMARY")
display(summary.round(2))

### 6.5.1 — Correct Hybrid Result Processing

Extract destination names according to the actual structure
returned by hybrid_graph_vector_retrieval().

Use metadata for ChromaDB results and graph_data for
Neo4j-only results.

In [ ]:
hybrid_evaluation_results = []

for case in evaluation_cases:

    question = case["question"]

    scope = validate_geographical_scope(question)

    retrieval_question = prepare_retrieval_question(
        question,
        scope["detected_country"]
    )

    results = hybrid_graph_vector_retrieval(
        query=retrieval_question,
        graph_keyword=case["graph_keyword"],
        top_k=3
    )

    expected = normalize_destination_name(
        case["expected_destination"]
    )

    retrieved_names = []

    for item in results:

        if "metadata" in item:
            name = item["metadata"].get("name", "")

        elif "graph_data" in item:
            name = item["graph_data"]["destination"]

        else:
            continue

        retrieved_names.append(name)

    relevant_count = sum(
        normalize_destination_name(name) == expected
        for name in retrieved_names
    )

    precision = (
        relevant_count / len(retrieved_names)
        if retrieved_names else 0
    )

    recall = min(relevant_count, 1)

    hybrid_evaluation_results.append({
        "question": question,
        "retrieved": retrieved_names,
        "precision_at_k": precision,
        "recall_at_k": recall
    })

    print("Question:", question)
    print("Retrieved:", retrieved_names)
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print("-" * 60)

### 6.5.3 — Compare Retrieval Systems

Compare ChromaDB, Neo4j and Hybrid Retrieval using their
average precision and recall.

This comparison helps identify the strengths of each
retrieval strategy.

In [ ]:
hybrid_df = pd.DataFrame(hybrid_evaluation_results)

hybrid_summary = hybrid_df.agg({
    "precision_at_k": "mean",
    "recall_at_k": "mean"
}).to_frame().T

hybrid_summary.insert(0, "system", "Hybrid")

comparison = pd.concat([
    summary,
    hybrid_summary.rename(columns={
        "precision_at_k": "average_precision",
        "recall_at_k": "average_recall"
    })
], ignore_index=True)

print("RETRIEVAL SYSTEM COMPARISON")

display(comparison.round(2))

### 6.5.4 — Standardize Retrieval Metrics

Use the top three unique destinations from each retrieval system.

Calculate Precision@3 and Recall@3 consistently and display
all results as percentages.

In [ ]:
hybrid_precision = []
hybrid_recall = []

for case, result in zip(evaluation_cases, hybrid_evaluation_results):

    expected = normalize_destination_name(
        case["expected_destination"]
    )

    top_three = result["retrieved"][:3]

    relevant_count = sum(
        normalize_destination_name(name) == expected
        for name in top_three
    )

    hybrid_precision.append(relevant_count / 3)
    hybrid_recall.append(min(relevant_count, 1))

comparison.loc[
    comparison["system"] == "Hybrid",
    "average_precision"
] = sum(hybrid_precision) / len(hybrid_precision) * 100

comparison.loc[
    comparison["system"] == "Hybrid",
    "average_recall"
] = sum(hybrid_recall) / len(hybrid_recall) * 100

display(comparison.round(2))

### 6.6.1 — Answer Accuracy and Hallucination Evaluation

Evaluate Qwen3-generated answers using four criteria:

1. Destination Accuracy
   - Are recommended destinations present in the retrieved context?

2. Factual Grounding
   - Are activities, costs, locations and other claims supported
     by the retrieved context?

3. Hallucination Detection
   - Does the model introduce unsupported information?

4. Geographical Compliance
   - Does the assistant respect its Rwanda-only restriction?

Answers must be evaluated against retrieved evidence,
not general model knowledge.

### 6.6.2 — Generate Answers for Evaluation

Run all five predefined evaluation questions through the
complete Rwanda Tourism Assistant workflow.

Store the generated answers together with their questions
and geographical validation results.

These answers will be reviewed against the retrieved context
to identify unsupported claims and hallucinations.

In [ ]:
answer_evaluation_results = []

for case in evaluation_cases:

    initial_state = {
        "question": case["question"],
        "graph_keyword": case["graph_keyword"],
        "in_scope": True,
        "detected_country": "",
        "retrieval_question": "",
        "vector_results": [],
        "graph_results": [],
        "final_answer": ""
    }

    result = tourism_agent.invoke(initial_state)

    answer_evaluation_results.append({
        "question": case["question"],
        "answer": result["final_answer"],
        "in_scope": result["in_scope"],
        "detected_country": result["detected_country"],
        "vector_results": result["vector_results"],
        "graph_results": result["graph_results"]
    })

    print("QUESTION:", case["question"])
    print("ANSWER:")
    print(result["final_answer"])
    print("=" * 80)

print("Completed answer evaluation:", len(answer_evaluation_results))

### 6.6.3 Hallucination Evaluation

This evaluation examines whether the generated answers are supported by the retrieved knowledge.

Each answer is reviewed at claim level and classified as:

- Grounded: Supported by retrieved information.
- Unsupported: Not established by the available evidence.
- Contradicted: Conflicts with retrieved information.

The evaluation focuses on factual grounding, unsupported details, and contradictions.

This is a manual evidence-based review of the five test answers, not an automated factuality score.

In [ ]:
import pandas as pd

hallucination_reviews = [
    {
        "question": "Gorilla and nature",
        "claim": "Volcanoes National Park is in Northern Province.",
        "classification": "Grounded",
        "reason": "Consistent with the destination information in our dataset."
    },
    {
        "question": "Gorilla and nature",
        "claim": "Gorilla trekking takes about one hour.",
        "classification": "Unsupported",
        "reason": "The retrieved information does not establish this duration."
    },
    {
        "question": "Gorilla and nature",
        "claim": "The gorilla permit costs USD 1,500 and should be booked 3–6 months ahead.",
        "classification": "Unsupported",
        "reason": "These specific price and booking claims are not established by the retrieved context."
    },
    {
        "question": "Gorilla and nature",
        "claim": "The journey from Kigali takes about two hours.",
        "classification": "Unsupported",
        "reason": "The exact travel time is not established by the retrieved context."
    },
    {
        "question": "Volcanoes district",
        "claim": "Volcanoes National Park is located in Musanze District.",
        "classification": "Grounded",
        "reason": "Consistent with the graph retrieval result."
    },
    {
        "question": "Lakes and water activities",
        "claim": "Lake Kivu is not located in Rwanda.",
        "classification": "Contradicted",
        "reason": "Our dataset includes Lake Kivu destinations in Rwandan districts."
    },
    {
        "question": "Lakes and water activities",
        "claim": "Karongi, Rubavu and Rusizi are in the DRC or outside Rwanda.",
        "classification": "Contradicted",
        "reason": "The dataset identifies these Lake Kivu destinations within Rwanda."
    },
    {
        "question": "Lakes and water activities",
        "claim": "Lake Muhazi is the only suitable lake for water activities.",
        "classification": "Unsupported",
        "reason": "The retrieved context does not establish that it is the only suitable lake."
    },
    {
        "question": "Cultural tourism",
        "claim": "The Ethnographic Museum is in Huye, Southern Province.",
        "classification": "Grounded",
        "reason": "Consistent with the destination information in our dataset."
    },
    {
        "question": "Cultural tourism",
        "claim": "The museum has over 10,000 artefacts and was inaugurated in 1989.",
        "classification": "Unsupported",
        "reason": "These specific historical and numerical details are not established by the retrieved context."
    },
    {
        "question": "Cultural tourism",
        "claim": "The museum is approximately 132 km from Kigali.",
        "classification": "Unsupported",
        "reason": "The exact distance is not established by the retrieved context."
    },
    {
        "question": "Tanzania gorilla trekking",
        "claim": "The assistant only provides tourism information about Rwanda.",
        "classification": "Grounded",
        "reason": "The geographical validation correctly rejected the Tanzania request."
    }
]

hallucination_df = pd.DataFrame(hallucination_reviews)

display(hallucination_df)

print("\nHALLUCINATION EVALUATION SUMMARY")

summary = (
    hallucination_df
    .groupby("classification")
    .size()
    .reindex(
        ["Grounded", "Unsupported", "Contradicted"],
        fill_value=0
    )
)

display(summary.to_frame("Number of Claims"))

print("Total claims reviewed:", len(hallucination_df))

## 7.1 Installing Gradio

Gradio will provide the web-based interface for our Rwanda Tourism Assistant.

We will use it to create an interactive chat interface connected to our existing LangGraph tourism agent.


In [ ]:
%pip install -q gradio

import gradio as gr

print("Gradio version:", gr.__version__)
print("Gradio successfully installed!")

### 7.2 Creating the Rwanda Tourism Assistant Chat Interface

We will create a conversational interface using Gradio ChatInterface.

The interface will:
- Accept tourism questions from users.
- Send questions to our existing LangGraph tourism agent.
- Display the generated answers.
- Maintain conversation history.

The existing Qwen3 model and retrieval systems will remain responsible for generating responses.

In [ ]:
import gradio as gr

# --------------------------------------------------
# 1. Chatbot Function
# --------------------------------------------------

def tourism_chat(message, history):

    initial_state = {
        "question": message,
        "graph_keyword": message,
        "in_scope": True,
        "detected_country": "",
        "retrieval_question": "",
        "vector_results": [],
        "graph_results": [],
        "final_answer": ""
    }

    result = tourism_agent.invoke(initial_state)

    return result["final_answer"]


# --------------------------------------------------
# 2. Create Gradio Chat Interface
# --------------------------------------------------

demo = gr.ChatInterface(
    fn=tourism_chat,

    title="🇷🇼 Rwanda Tourism Assistant",

    description=(
        "Your AI-powered travel companion for exploring Rwanda. "
        "Ask about destinations, wildlife, culture, nature, "
        "lakes and travel experiences."
    ),

    examples=[
        "Where can I see gorillas in Rwanda?",
        "Recommend lakes for water activities.",
        "Which district is Volcanoes National Park located in?",
        "Where can I experience cultural tourism?",
        "Recommend destinations for nature lovers."
    ],

    textbox=gr.Textbox(
        placeholder="Ask me about tourism in Rwanda...",
        container=False,
        scale=7
    )
)


# --------------------------------------------------
# 3. Launch in External Browser
# --------------------------------------------------

demo.launch(
    share=True,
    inline=False,
    debug=True,
    show_error=True
)

### Upload Rwanda Tourism Hero Image

Upload the generated Rwanda landscape image into Google Colab
and save it under the expected filename: rwanda_hero.png.

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

if uploaded:
    uploaded_filename = next(iter(uploaded))

    shutil.move(
        uploaded_filename,
        "/content/rwanda_hero.png"
    )

    print("Rwanda hero image uploaded successfully!")
    print("Location: /content/rwanda_hero.png")
else:
    print("No image was uploaded.")

## Rwanda Tourism Assistant — UI Design

Create a modern, responsive tourism chatbot interface using Gradio.
Keep the existing tourism_chat() function and its backend unchanged.

The interface includes Rwanda branding, a logo, welcome section,
suggested questions, and a professionally styled chat window.

In [ ]:
import gradio as gr
import os
import base64

# ==================================================
# 1. LOAD RWANDA LANDSCAPE IMAGE
# ==================================================

hero_image_path = "/content/rwanda_hero.png"

if not os.path.exists(hero_image_path):
    raise FileNotFoundError(
        "Please upload rwanda_hero.png to the Colab Files panel."
    )

with open(hero_image_path, "rb") as image_file:
    hero_base64 = base64.b64encode(
        image_file.read()
    ).decode("utf-8")


# ==================================================
# 2. CUSTOM CSS
# ==================================================

custom_css = """

/* FULL WIDTH RESPONSIVE APPLICATION */

.gradio-container {
    width: 100% !important;
    max-width: 100% !important;
    margin: 0 auto !important;
    padding: 16px clamp(12px, 3vw, 45px) !important;
    box-sizing: border-box !important;
    background: #f6faf7 !important;
}

.main,
.contain {
    max-width: 100% !important;
    width: 100% !important;
}

/* RWANDA HERO HEADER */

.tourism-hero {
    position: relative;
    width: 100%;
    min-height: 245px;
    overflow: hidden;
    border-radius: 22px;
    display: flex;
    justify-content: center;
    align-items: center;
    text-align: center;
    margin-bottom: 20px;
    isolation: isolate;
    background: #075b3b;
}

/* ANIMATED LANDSCAPE */

.tourism-hero::before {
    content: "";
    position: absolute;
    inset: -5%;
    z-index: -2;

    background-image: url("data:image/png;base64,REPLACE_HERO_IMAGE");
    background-size: cover;
    background-position: center 52%;
    background-repeat: no-repeat;

    animation: landscapeMotion 28s ease-in-out infinite alternate;
}

/* TRANSPARENT GREEN OVERLAY */

.tourism-hero::after {
    content: "";
    position: absolute;
    inset: 0;
    z-index: -1;

    background: linear-gradient(
        110deg,
        rgba(0, 69, 42, 0.76),
        rgba(5, 103, 62, 0.48),
        rgba(0, 55, 42, 0.65)
    );
}

/* BACKGROUND ANIMATION */

@keyframes landscapeMotion {

    0% {
        transform: scale(1);
        background-position: 45% center;
    }

    100% {
        transform: scale(1.12);
        background-position: 58% center;
    }

}

/* HERO CONTENT */

.hero-content {
    position: relative;
    z-index: 2;
    padding: 35px 20px;
    color: white;
}

.hero-logo {
    font-size: 45px;
    margin-bottom: 5px;
    filter: drop-shadow(0 2px 5px rgba(0,0,0,0.3));
}

.hero-content h1 {
    color: white !important;
    font-size: clamp(25px, 3.5vw, 42px) !important;
    font-weight: 800 !important;
    margin: 8px 0 !important;
    text-shadow: 0 2px 8px rgba(0,0,0,0.25);
}

.hero-content p {
    color: white !important;
    font-size: clamp(13px, 1.4vw, 17px);
    max-width: 850px;
    margin: 0 auto;
    line-height: 1.7;
    text-shadow: 0 1px 5px rgba(0,0,0,0.3);
}

/* SUGGESTED QUESTIONS */

.suggestions-section {
    background: white;
    border: 1px solid #e1eae3;
    border-radius: 16px;
    padding: 16px;
    margin-bottom: 14px;
    width: 100%;
    box-sizing: border-box;
}

.suggestions-section h3 {
    color: #176b45;
    font-size: 16px;
    margin-bottom: 12px;
}

.suggestion-card {
    min-width: 0 !important;
}

.suggestion-card button {
    min-height: 65px !important;
    height: auto !important;
    white-space: normal !important;
    text-align: left !important;
    background: #f4f8f5 !important;
    border: 1px solid #dce9df !important;
    border-radius: 12px !important;
    color: #24583d !important;
    transition: all 0.2s ease !important;
}

.suggestion-card button:hover {
    background: #e8f4ec !important;
    border-color: #168451 !important;
    transform: translateY(-2px);
}

/* CHAT AREA */

#tourism-chat {
    width: 100% !important;
    min-height: 120px !important;
    height: auto !important;
    background: white !important;
    border: 1px solid #e1eae3 !important;
    border-radius: 16px !important;
}

/* CHAT MESSAGE AREA */

#tourism-chat .bubble-wrap {
    max-height: none !important;
}

/* INPUT */

#tourism-input textarea {
    border-radius: 12px !important;
    border: 1px solid #d5e4da !important;
    background: white !important;
}

/* SEND BUTTON */

#send-button {
    min-width: 110px !important;
    border-radius: 12px !important;
    background: #087747 !important;
    color: white !important;
    font-weight: 600 !important;
    transition: all 0.2s ease !important;
}

#send-button:hover {
    background: #065c37 !important;
    transform: translateY(-2px);
}

/* FOOTER */

.tourism-footer {
    text-align: center;
    color: #718078;
    font-size: 12px;
    padding: 18px;
}

footer {
    display: none !important;
}

/* MOBILE RESPONSIVENESS */

@media (max-width: 768px) {

    .gradio-container {
        padding: 10px !important;
    }

    .tourism-hero {
        min-height: 210px;
        border-radius: 15px;
    }

    .hero-content {
        padding: 25px 12px;
    }

    .hero-logo {
        font-size: 35px;
    }

    .suggestions-section {
        padding: 10px;
    }

    .suggestion-card button {
        min-height: 58px !important;
        font-size: 12px !important;
    }

    #send-button {
        min-width: 75px !important;
    }

}

/* ACCESSIBILITY */

@media (prefers-reduced-motion: reduce) {

    .tourism-hero::before {
        animation: none;
    }

}

"""


# Insert image into CSS
custom_css = custom_css.replace(
    "REPLACE_HERO_IMAGE",
    hero_base64
)


# ==================================================
# 3. HERO HEADER
# ==================================================

hero_header = """

<div class="tourism-hero">

    <div class="hero-content">

        <div class="hero-logo">🇷🇼</div>

        <h1>Rwanda Tourism Assistant</h1>

        <p>
            Discover the Land of a Thousand Hills.
            Explore Rwanda's beautiful landscapes, wildlife,
            culture and unforgettable destinations.
        </p>

    </div>

</div>

"""


# ==================================================
# 4. CHAT MESSAGE CALLBACK
# ==================================================

def submit_message(message, history):

    if not message or not message.strip():
        return "", history or []

    history = history or []

    # Call existing AI backend
    answer = tourism_chat(message, history)

    # Append conversation messages
    history = history + [
        {
            "role": "user",
            "content": message
        },
        {
            "role": "assistant",
            "content": answer
        }
    ]

    return "", history


# ==================================================
# 5. CREATE GRADIO INTERFACE
# ==================================================

with gr.Blocks(
    title="Rwanda Tourism Assistant",
    theme=gr.themes.Soft(
        primary_hue="green",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    css=custom_css
) as demo:

    # ----------------------------------------------
    # HERO HEADER
    # ----------------------------------------------

    gr.HTML(hero_header)

    # ----------------------------------------------
    # SUGGESTED QUESTIONS
    # ----------------------------------------------

    with gr.Column(elem_classes="suggestions-section"):

        gr.Markdown("### ✨ Suggested questions")

        with gr.Row():

            suggestion_1 = gr.Button(
                "🦍 Where can I see gorillas in Rwanda?",
                elem_classes="suggestion-card"
            )

            suggestion_2 = gr.Button(
                "🏞️ Recommend lakes for water activities.",
                elem_classes="suggestion-card"
            )

        with gr.Row():

            suggestion_3 = gr.Button(
                "📍 Where is Volcanoes National Park located?",
                elem_classes="suggestion-card"
            )

            suggestion_4 = gr.Button(
                "🎭 Where can I experience Rwandan culture?",
                elem_classes="suggestion-card"
            )

    # ----------------------------------------------
    # CHAT DISPLAY
    # ----------------------------------------------

    chatbot = gr.Chatbot(
        height=None,
        show_label=False,
        elem_id="tourism-chat",
        placeholder="Your Rwanda travel conversation starts here..."
    )

    # ----------------------------------------------
    # MESSAGE INPUT AND SEND BUTTON
    # ----------------------------------------------

    with gr.Row():

        message_input = gr.Textbox(
            placeholder="Ask me about tourism in Rwanda...",
            show_label=False,
            lines=1,
            max_lines=4,
            scale=8,
            elem_id="tourism-input"
        )

        send_button = gr.Button(
            "➤ Send",
            variant="primary",
            scale=1,
            elem_id="send-button"
        )

    # ----------------------------------------------
    # FOOTER
    # ----------------------------------------------

    gr.HTML("""
        <div class="tourism-footer">
            🇷🇼 Rwanda Tourism Assistant · Powered by AI
            <br>
            Discover. Explore. Experience Rwanda.
        </div>
    """)

    # ----------------------------------------------
    # CONNECT SEND BUTTON
    # ----------------------------------------------

    send_button.click(
        fn=submit_message,
        inputs=[message_input, chatbot],
        outputs=[message_input, chatbot]
    )

    # Submit using Enter
    message_input.submit(
        fn=submit_message,
        inputs=[message_input, chatbot],
        outputs=[message_input, chatbot]
    )

    # ----------------------------------------------
    # CONNECT SUGGESTED QUESTIONS
    # ----------------------------------------------

    suggestions = [

        (
            suggestion_1,
            "Where can I see gorillas in Rwanda?"
        ),

        (
            suggestion_2,
            "Recommend lakes for water activities."
        ),

        (
            suggestion_3,
            "Which district is Volcanoes National Park located in?"
        ),

        (
            suggestion_4,
            "Where can I experience cultural tourism?"
        )

    ]

    for button, question in suggestions:

        button.click(
            fn=lambda q=question: q,
            inputs=None,
            outputs=message_input
        )


# ==================================================
# 6. LAUNCH APPLICATION
# ==================================================

demo.queue()

demo.launch(
    share=True,
    inline=False,
    debug=True,
    show_error=True
)